# Infaction — Faction, Naming & Zone Generator MVP v0.4.1 (LOCKED)

**Active Zone selection:** 36 Scale 4–5 Types; A + Soft MAX; bonus 0.75, penalty 1.0 per active penalty term; Function count threshold >=75; 9 additive Zone score modifiers.

**Heldout QA:** 10,000 seed worlds (14,900–24,899), all Zone types between 5.33% and 38.85%; Function minimum 99.95%, maximum 100% (empirical, not hard guaranteed). No per-world frequency enforcement.

Faction and naming generator remain unchanged. This standalone notebook is separate from the existing Rantau live website World v0.11.0.



In [ ]:
import json, random, math, csv, pathlib
from collections import Counter
from IPython.display import display, Markdown

VERSION = '0.5.0'
# Self-contained Book1 snapshot; all affinity numbers and Grammar weights come from Book1.xlsx.
BOOK1 = json.loads('{"source":"Book1.xlsx (2026-11-20, exact snapshot)","types":["Government","Military","Religious","Criminal","Revolutionary","Conspiracy","Merchant","Clan","Cultural","Outsider","Elite","Poor","Academic"],"animal":{"Lion":{"Government":94.0,"Military":54.08,"Religious":46.38,"Criminal":5.0,"Revolutionary":10.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":65.62,"Cultural":0.0,"Outsider":0.0,"Elite":86.84,"Poor":0.0,"Academic":0.0},"Eagle":{"Government":94.0,"Military":79.94,"Religious":0.0,"Criminal":0.0,"Revolutionary":50.75,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":46.38,"Elite":79.29,"Poor":0.0,"Academic":10.0},"Tiger":{"Government":79.94,"Military":94.0,"Religious":0.0,"Criminal":15.0,"Revolutionary":54.08,"Conspiracy":0.0,"Merchant":0.0,"Clan":46.38,"Cultural":0.0,"Outsider":0.0,"Elite":68.48,"Poor":0.0,"Academic":0.0},"Bear":{"Government":71.97,"Military":94.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":50.75,"Conspiracy":0.0,"Merchant":0.0,"Clan":91.65,"Cultural":46.38,"Outsider":0.0,"Elite":0.0,"Poor":10.0,"Academic":0.0},"Bull":{"Government":84.99,"Military":74.68,"Religious":0.0,"Criminal":0.0,"Revolutionary":15.0,"Conspiracy":0.0,"Merchant":69.19,"Clan":50.75,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":46.38,"Academic":0.0},"Elephant":{"Government":87.45,"Military":0.0,"Religious":50.75,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":46.38,"Clan":74.01,"Cultural":79.94,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":20.0},"Dragon":{"Government":91.65,"Military":54.08,"Religious":84.37,"Criminal":10.0,"Revolutionary":46.38,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":50.75,"Poor":0.0,"Academic":0.0},"Stag":{"Government":46.38,"Military":0.0,"Religious":54.08,"Criminal":0.0,"Revolutionary":5.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":89.26,"Cultural":94.0,"Outsider":0.0,"Elite":50.75,"Poor":0.0,"Academic":0.0},"Wolf":{"Government":0.0,"Military":81.85,"Religious":0.0,"Criminal":50.75,"Revolutionary":94.0,"Conspiracy":15.0,"Merchant":0.0,"Clan":79.29,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":46.38,"Academic":0.0},"Fox":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":81.85,"Revolutionary":46.38,"Conspiracy":54.08,"Merchant":94.0,"Clan":0.0,"Cultural":0.0,"Outsider":50.75,"Elite":0.0,"Poor":0.0,"Academic":15.0},"Serpent":{"Government":46.38,"Military":0.0,"Religious":89.26,"Criminal":54.08,"Revolutionary":0.0,"Conspiracy":91.65,"Merchant":10.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":65.62},"Leopard":{"Government":0.0,"Military":89.86,"Religious":0.0,"Criminal":54.08,"Revolutionary":79.94,"Conspiracy":0.0,"Merchant":0.0,"Clan":46.38,"Cultural":0.0,"Outsider":0.0,"Elite":50.75,"Poor":0.0,"Academic":0.0},"Crocodile":{"Government":69.19,"Military":58.08,"Religious":0.0,"Criminal":87.45,"Revolutionary":0.0,"Conspiracy":50.75,"Merchant":0.0,"Clan":0.0,"Cultural":46.38,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"Jackal":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":94.0,"Revolutionary":77.34,"Conspiracy":0.0,"Merchant":46.38,"Clan":0.0,"Cultural":0.0,"Outsider":71.97,"Elite":0.0,"Poor":50.75,"Academic":0.0},"Cat":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":50.75,"Revolutionary":0.0,"Conspiracy":82.49,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":76.68,"Elite":84.37,"Poor":0.0,"Academic":46.38},"Shark":{"Government":0.0,"Military":54.08,"Religious":0.0,"Criminal":77.34,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":89.86,"Clan":0.0,"Cultural":0.0,"Outsider":46.38,"Elite":50.75,"Poor":0.0,"Academic":0.0},"Raven":{"Government":0.0,"Military":0.0,"Religious":54.08,"Criminal":46.38,"Revolutionary":50.75,"Conspiracy":94.0,"Merchant":0.0,"Clan":0.0,"Cultural":15.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":86.84},"Owl":{"Government":50.75,"Military":0.0,"Religious":54.08,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":77.34,"Merchant":0.0,"Clan":0.0,"Cultural":15.0,"Outsider":0.0,"Elite":46.38,"Poor":0.0,"Academic":94.0},"Spider":{"Government":5.0,"Military":0.0,"Religious":0.0,"Criminal":79.94,"Revolutionary":46.38,"Conspiracy":94.0,"Merchant":71.97,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":50.75},"Bat":{"Government":0.0,"Military":0.0,"Religious":46.38,"Criminal":74.68,"Revolutionary":0.0,"Conspiracy":89.86,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":50.75,"Elite":0.0,"Poor":0.0,"Academic":71.97},"Scorpion":{"Government":0.0,"Military":54.08,"Religious":46.38,"Criminal":89.86,"Revolutionary":50.75,"Conspiracy":71.97,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"Moth":{"Government":0.0,"Military":0.0,"Religious":54.08,"Criminal":0.0,"Revolutionary":50.75,"Conspiracy":87.45,"Merchant":0.0,"Clan":0.0,"Cultural":46.38,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":84.37},"Octopus":{"Government":46.38,"Military":0.0,"Religious":0.0,"Criminal":50.75,"Revolutionary":0.0,"Conspiracy":74.68,"Merchant":91.65,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":76.68},"Phoenix":{"Government":54.08,"Military":0.0,"Religious":91.65,"Criminal":0.0,"Revolutionary":94.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":50.75,"Outsider":0.0,"Elite":46.38,"Poor":0.0,"Academic":0.0},"Bee":{"Government":15.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":94.0,"Clan":50.75,"Cultural":54.08,"Outsider":0.0,"Elite":0.0,"Poor":79.94,"Academic":46.38},"Ant":{"Government":10.0,"Military":54.08,"Religious":0.0,"Criminal":0.0,"Revolutionary":81.85,"Conspiracy":0.0,"Merchant":50.75,"Clan":0.0,"Cultural":46.38,"Outsider":0.0,"Elite":0.0,"Poor":94.0,"Academic":0.0},"Rat":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":94.0,"Revolutionary":54.08,"Conspiracy":15.0,"Merchant":46.38,"Clan":0.0,"Cultural":0.0,"Outsider":50.75,"Elite":0.0,"Poor":81.85,"Academic":0.0},"Goat":{"Government":0.0,"Military":0.0,"Religious":50.75,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":71.97,"Cultural":74.68,"Outsider":46.38,"Elite":0.0,"Poor":94.0,"Academic":0.0},"Sheep":{"Government":0.0,"Military":0.0,"Religious":77.34,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":46.38,"Clan":50.75,"Cultural":54.08,"Outsider":0.0,"Elite":0.0,"Poor":91.65,"Academic":0.0},"Boar":{"Government":0.0,"Military":71.97,"Religious":0.0,"Criminal":0.0,"Revolutionary":91.65,"Conspiracy":0.0,"Merchant":0.0,"Clan":76.68,"Cultural":46.38,"Outsider":0.0,"Elite":0.0,"Poor":50.75,"Academic":0.0},"Dog":{"Government":46.38,"Military":58.08,"Religious":0.0,"Criminal":5.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":94.0,"Cultural":0.0,"Outsider":65.62,"Elite":0.0,"Poor":71.97,"Academic":0.0},"Camel":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":81.85,"Clan":50.75,"Cultural":54.08,"Outsider":94.0,"Elite":0.0,"Poor":46.38,"Academic":0.0},"Horse":{"Government":0.0,"Military":58.08,"Religious":0.0,"Criminal":0.0,"Revolutionary":46.38,"Conspiracy":0.0,"Merchant":74.01,"Clan":0.0,"Cultural":50.75,"Outsider":94.0,"Elite":10.0,"Poor":0.0,"Academic":0.0},"Falcon":{"Government":54.08,"Military":91.65,"Religious":0.0,"Criminal":0.0,"Revolutionary":84.37,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":46.38,"Elite":65.62,"Poor":0.0,"Academic":0.0},"Condor":{"Government":54.08,"Military":0.0,"Religious":46.38,"Criminal":0.0,"Revolutionary":74.68,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":84.99,"Outsider":50.75,"Elite":0.0,"Poor":0.0,"Academic":0.0},"Parrot":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":50.75,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":79.94,"Clan":0.0,"Cultural":54.08,"Outsider":89.86,"Elite":0.0,"Poor":0.0,"Academic":46.38},"Butterfly":{"Government":0.0,"Military":0.0,"Religious":74.68,"Criminal":0.0,"Revolutionary":89.86,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":54.08,"Outsider":50.75,"Elite":0.0,"Poor":0.0,"Academic":46.38},"Hare":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":54.08,"Conspiracy":0.0,"Merchant":46.38,"Clan":0.0,"Cultural":50.75,"Outsider":87.45,"Elite":0.0,"Poor":77.34,"Academic":0.0},"Swallow":{"Government":0.0,"Military":0.0,"Religious":46.38,"Criminal":0.0,"Revolutionary":71.97,"Conspiracy":0.0,"Merchant":54.08,"Clan":0.0,"Cultural":50.75,"Outsider":91.65,"Elite":0.0,"Poor":0.0,"Academic":0.0},"Dolphin":{"Government":0.0,"Military":0.0,"Religious":46.38,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":87.45,"Clan":0.0,"Cultural":54.08,"Outsider":81.85,"Elite":0.0,"Poor":0.0,"Academic":50.75},"Swan":{"Government":10.0,"Military":0.0,"Religious":81.85,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":50.75,"Cultural":54.08,"Outsider":0.0,"Elite":94.0,"Poor":0.0,"Academic":46.38},"Peacock":{"Government":77.34,"Military":0.0,"Religious":50.75,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":54.08,"Clan":0.0,"Cultural":46.38,"Outsider":0.0,"Elite":94.0,"Poor":0.0,"Academic":0.0},"Dove":{"Government":50.75,"Military":0.0,"Religious":94.0,"Criminal":0.0,"Revolutionary":58.08,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":54.08,"Outsider":0.0,"Elite":0.0,"Poor":46.38,"Academic":0.0},"Crane":{"Government":0.0,"Military":0.0,"Religious":86.84,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":46.38,"Cultural":94.0,"Outsider":0.0,"Elite":50.75,"Poor":0.0,"Academic":74.01},"Turtle":{"Government":0.0,"Military":0.0,"Religious":54.08,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":84.37,"Cultural":91.65,"Outsider":0.0,"Elite":0.0,"Poor":46.38,"Academic":50.75},"Rooster":{"Government":0.0,"Military":46.38,"Religious":50.75,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":86.84,"Cultural":87.45,"Outsider":0.0,"Elite":0.0,"Poor":54.08,"Academic":0.0},"Unicorn":{"Government":54.08,"Military":0.0,"Religious":79.94,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":50.75,"Outsider":0.0,"Elite":91.65,"Poor":0.0,"Academic":46.38},"Whale":{"Government":54.08,"Military":0.0,"Religious":71.97,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":50.75,"Clan":0.0,"Cultural":89.86,"Outsider":46.38,"Elite":0.0,"Poor":0.0,"Academic":0.0},"Monkey":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":71.97,"Revolutionary":54.08,"Conspiracy":10.0,"Merchant":0.0,"Clan":0.0,"Cultural":50.75,"Outsider":46.38,"Elite":0.0,"Poor":0.0,"Academic":94.0},"Frog":{"Government":0.0,"Military":0.0,"Religious":54.08,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":71.97,"Outsider":46.38,"Elite":0.0,"Poor":87.45,"Academic":50.75},"Jaguar":{"Government":46.38,"Military":87.45,"Religious":50.75,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":77.34,"Outsider":0.0,"Elite":74.01,"Poor":0.0,"Academic":0.0},"Lizard":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":50.75,"Revolutionary":0.0,"Conspiracy":84.99,"Merchant":0.0,"Clan":0.0,"Cultural":46.38,"Outsider":74.01,"Elite":0.0,"Poor":0.0,"Academic":81.85},"Mantis":{"Government":0.0,"Military":54.08,"Religious":46.38,"Criminal":50.75,"Revolutionary":0.0,"Conspiracy":58.08,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":91.65},"Beetle":{"Government":0.0,"Military":0.0,"Religious":54.08,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":46.38,"Cultural":58.08,"Outsider":0.0,"Elite":0.0,"Poor":89.86,"Academic":50.75},"Hyena":{"Government":0.0,"Military":50.75,"Religious":0.0,"Criminal":91.65,"Revolutionary":58.08,"Conspiracy":10.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":46.38,"Elite":0.0,"Poor":69.19,"Academic":0.0},"Griffin":{"Government":89.86,"Military":77.34,"Religious":50.75,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":46.38,"Cultural":0.0,"Outsider":0.0,"Elite":76.68,"Poor":0.0,"Academic":0.0}},"base":{"Empire":{"affinity":{"Government":94.0,"Military":34.31,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":57.57,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Kingdom":{"affinity":{"Government":91.91,"Military":34.31,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":60.59,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Republic":{"affinity":{"Government":89.79,"Military":34.31,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":38.79,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Dominion":{"affinity":{"Government":84.84,"Military":34.31,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":38.79,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Council":{"affinity":{"Government":83.14,"Military":0.0,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":84.28,"Poor":0.0,"Academic":34.31},"grammar":{"M":100.0,"O":50.0,"P":50.0,"S":50.0}},"Court":{"affinity":{"Government":63.49,"Military":0.0,"Religious":0.0,"Criminal":27.92,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":34.31,"Cultural":0.0,"Outsider":0.0,"Elite":89.79,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Commonwealth":{"affinity":{"Government":85.4,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":27.92,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":38.79,"Outsider":0.0,"Elite":0.0,"Poor":34.31,"Academic":0.0},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Directorate":{"affinity":{"Government":87.62,"Military":34.31,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":74.17,"Poor":0.0,"Academic":54.4},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Ministry":{"affinity":{"Government":70.32,"Military":0.0,"Religious":60.59,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":27.92,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":34.31},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Authority":{"affinity":{"Government":80.83,"Military":38.79,"Religious":34.31,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":27.92},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Legion":{"affinity":{"Government":38.79,"Military":94.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":34.31,"Conspiracy":0.0,"Merchant":0.0,"Clan":27.92,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Guard":{"affinity":{"Government":38.79,"Military":89.79,"Religious":0.0,"Criminal":0.0,"Revolutionary":34.31,"Conspiracy":0.0,"Merchant":0.0,"Clan":27.92,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Army":{"affinity":{"Government":38.79,"Military":91.91,"Religious":0.0,"Criminal":0.0,"Revolutionary":34.31,"Conspiracy":0.0,"Merchant":0.0,"Clan":27.92,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Militia":{"affinity":{"Government":27.92,"Military":85.4,"Religious":0.0,"Criminal":0.0,"Revolutionary":69.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":52.77,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Vanguard":{"affinity":{"Government":38.79,"Military":83.14,"Religious":0.0,"Criminal":0.0,"Revolutionary":34.31,"Conspiracy":0.0,"Merchant":0.0,"Clan":27.92,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Watch":{"affinity":{"Government":38.79,"Military":80.83,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":27.92,"Merchant":0.0,"Clan":34.31,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Corps":{"affinity":{"Government":34.31,"Military":87.62,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":27.92,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":38.79},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Wardens":{"affinity":{"Government":0.0,"Military":77.86,"Religious":34.31,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":38.79,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":27.92},"grammar":{"M":100.0,"O":50.0,"P":50.0,"S":0.0}},"Rangers":{"affinity":{"Government":0.0,"Military":75.42,"Religious":0.0,"Criminal":0.0,"Revolutionary":27.92,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":34.31,"Outsider":38.79,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Order":{"affinity":{"Government":0.0,"Military":59.09,"Religious":94.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":81.99,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":60.59},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":0.0}},"Covenant":{"affinity":{"Government":0.0,"Military":0.0,"Religious":91.91,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":27.92,"Merchant":0.0,"Clan":66.29,"Cultural":34.31,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":0.0}},"Faith":{"affinity":{"Government":0.0,"Military":0.0,"Religious":89.79,"Criminal":0.0,"Revolutionary":27.92,"Conspiracy":38.79,"Merchant":0.0,"Clan":0.0,"Cultural":34.31,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Sect":{"affinity":{"Government":0.0,"Military":0.0,"Religious":87.62,"Criminal":0.0,"Revolutionary":27.92,"Conspiracy":38.79,"Merchant":0.0,"Clan":0.0,"Cultural":34.31,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Brotherhood":{"affinity":{"Government":0.0,"Military":0.0,"Religious":78.99,"Criminal":0.0,"Revolutionary":50.13,"Conspiracy":56.14,"Merchant":0.0,"Clan":72.81,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":65.55,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":0.0}},"Fellowship":{"affinity":{"Government":0.0,"Military":0.0,"Religious":77.86,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":27.92,"Cultural":60.59,"Outsider":34.31,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":0.0}},"Chapter":{"affinity":{"Government":0.0,"Military":27.92,"Religious":75.42,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":79.65,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":72.91},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Disciples":{"affinity":{"Government":0.0,"Military":0.0,"Religious":80.83,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":34.31,"Merchant":0.0,"Clan":0.0,"Cultural":27.92,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":38.79},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":0.0}},"Cult":{"affinity":{"Government":0.0,"Military":0.0,"Religious":85.4,"Criminal":34.31,"Revolutionary":27.92,"Conspiracy":66.29,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":0.0}},"Guild":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":34.31,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":94.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":27.92,"Academic":38.79},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Company":{"affinity":{"Government":0.0,"Military":38.79,"Religious":0.0,"Criminal":69.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":91.91,"Clan":0.0,"Cultural":0.0,"Outsider":67.66,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Consortium":{"affinity":{"Government":27.92,"Military":0.0,"Religious":0.0,"Criminal":38.79,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":89.79,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":34.31,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Syndicate":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":94.0,"Revolutionary":0.0,"Conspiracy":34.31,"Merchant":79.65,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":27.92,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Cartel":{"affinity":{"Government":27.92,"Military":0.0,"Religious":0.0,"Criminal":91.91,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":38.79,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":34.31,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Cooperative":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":85.4,"Clan":0.0,"Cultural":34.31,"Outsider":27.92,"Elite":0.0,"Poor":84.28,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Corporation":{"affinity":{"Government":27.92,"Military":0.0,"Religious":0.0,"Criminal":38.79,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":87.62,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":34.31,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Chamber":{"affinity":{"Government":34.31,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":80.83,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":76.65,"Poor":0.0,"Academic":27.92},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Trust":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":75.42,"Revolutionary":0.0,"Conspiracy":49.36,"Merchant":77.86,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":66.29,"Poor":0.0,"Academic":27.92},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Clan":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":94.0,"Cultural":76.65,"Outsider":27.92,"Elite":0.0,"Poor":34.31,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":50.0}},"House":{"affinity":{"Government":34.31,"Military":0.0,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":91.91,"Cultural":0.0,"Outsider":0.0,"Elite":90.85,"Poor":0.0,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":50.0}},"Dynasty":{"affinity":{"Government":34.31,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":79.65,"Cultural":27.92,"Outsider":0.0,"Elite":91.91,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Family":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":63.49,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":89.79,"Cultural":34.31,"Outsider":0.0,"Elite":27.92,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Tribe":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":77.86,"Cultural":88.71,"Outsider":27.92,"Elite":34.31,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"People":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":60.59,"Cultural":94.0,"Outsider":59.09,"Elite":0.0,"Poor":74.17,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":0.0}},"Community":{"affinity":{"Government":0.0,"Military":0.0,"Religious":41.89,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":63.49,"Cultural":87.62,"Outsider":70.32,"Elite":0.0,"Poor":71.62,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Nation":{"affinity":{"Government":38.79,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":27.92,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":89.79,"Outsider":34.31,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Kindred":{"affinity":{"Government":0.0,"Military":0.0,"Religious":34.31,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":51.09,"Merchant":0.0,"Clan":87.62,"Cultural":81.99,"Outsider":45.75,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":0.0}},"Lineage":{"affinity":{"Government":0.0,"Military":0.0,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":85.4,"Cultural":64.91,"Outsider":0.0,"Elite":71.62,"Poor":0.0,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":0.0}},"Resistance":{"affinity":{"Government":0.0,"Military":34.31,"Religious":0.0,"Criminal":0.0,"Revolutionary":94.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":27.92,"Elite":0.0,"Poor":69.0,"Academic":0.0},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Movement":{"affinity":{"Government":0.0,"Military":27.92,"Religious":0.0,"Criminal":0.0,"Revolutionary":91.91,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":34.31,"Outsider":0.0,"Elite":0.0,"Poor":79.65,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Front":{"affinity":{"Government":0.0,"Military":27.92,"Religious":0.0,"Criminal":0.0,"Revolutionary":89.79,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":34.31,"Outsider":0.0,"Elite":0.0,"Poor":38.79,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Union":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":81.99,"Conspiracy":0.0,"Merchant":34.31,"Clan":0.0,"Cultural":27.92,"Outsider":0.0,"Elite":0.0,"Poor":94.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Collective":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":79.65,"Conspiracy":0.0,"Merchant":27.92,"Clan":0.0,"Cultural":59.09,"Outsider":0.0,"Elite":0.0,"Poor":91.91,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Alliance":{"affinity":{"Government":38.79,"Military":34.31,"Religious":0.0,"Criminal":0.0,"Revolutionary":80.83,"Conspiracy":0.0,"Merchant":27.92,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":50.0,"S":0.0}},"League":{"affinity":{"Government":34.31,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":74.17,"Conspiracy":0.0,"Merchant":75.42,"Clan":0.0,"Cultural":27.92,"Outsider":56.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":50.0,"S":0.0}},"Pact":{"affinity":{"Government":0.0,"Military":27.92,"Religious":0.0,"Criminal":0.0,"Revolutionary":75.42,"Conspiracy":52.77,"Merchant":30.61,"Clan":63.49,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":0.0}},"Liberation":{"affinity":{"Government":0.0,"Military":27.92,"Religious":0.0,"Criminal":0.0,"Revolutionary":87.62,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":34.31,"Outsider":0.0,"Elite":0.0,"Poor":66.29,"Academic":0.0},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Academy":{"affinity":{"Government":0.0,"Military":38.79,"Religious":34.31,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":27.92,"Poor":0.0,"Academic":94.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Institute":{"affinity":{"Government":38.79,"Military":0.0,"Religious":27.92,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":34.31,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":91.91},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"School":{"affinity":{"Government":0.0,"Military":34.31,"Religious":38.79,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":27.92,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":87.62},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Society":{"affinity":{"Government":0.0,"Military":0.0,"Religious":59.09,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":89.79,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":54.4,"Poor":0.0,"Academic":79.65},"grammar":{"M":100.0,"O":50.0,"P":50.0,"S":50.0}},"Circle":{"affinity":{"Government":0.0,"Military":0.0,"Religious":34.31,"Criminal":62.06,"Revolutionary":0.0,"Conspiracy":94.0,"Merchant":0.0,"Clan":27.92,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":74.17},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":50.0}},"Cabal":{"affinity":{"Government":27.92,"Military":0.0,"Religious":0.0,"Criminal":86.52,"Revolutionary":0.0,"Conspiracy":91.91,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":34.31,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Conclave":{"affinity":{"Government":27.92,"Military":0.0,"Religious":66.29,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":87.62,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":64.91,"Poor":0.0,"Academic":45.75},"grammar":{"M":50.0,"O":100.0,"P":0.0,"S":0.0}},"Scholars":{"affinity":{"Government":0.0,"Military":0.0,"Religious":38.79,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":34.31,"Merchant":0.0,"Clan":0.0,"Cultural":27.92,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":85.4},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":0.0}},"Collegium":{"affinity":{"Government":27.92,"Military":0.0,"Religious":38.79,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":34.31,"Poor":0.0,"Academic":89.79},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Keepers":{"affinity":{"Government":0.0,"Military":0.0,"Religious":69.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":80.83,"Merchant":0.0,"Clan":0.0,"Cultural":27.92,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":67.66},"grammar":{"M":50.0,"O":100.0,"P":50.0,"S":0.0}},"Caravan":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":66.29,"Clan":27.92,"Cultural":34.31,"Outsider":89.79,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Fleet":{"affinity":{"Government":27.92,"Military":64.91,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":34.31,"Clan":0.0,"Cultural":0.0,"Outsider":79.65,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Band":{"affinity":{"Government":0.0,"Military":52.77,"Religious":0.0,"Criminal":84.28,"Revolutionary":27.92,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":80.83,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":50.0}},"Enclave":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":27.92,"Merchant":0.0,"Clan":45.75,"Cultural":66.29,"Outsider":85.4,"Elite":34.31,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Commune":{"affinity":{"Government":0.0,"Military":0.0,"Religious":27.92,"Criminal":0.0,"Revolutionary":34.31,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":57.57,"Outsider":0.0,"Elite":0.0,"Poor":89.79,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Network":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":81.99,"Revolutionary":0.0,"Conspiracy":77.86,"Merchant":56.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":27.92},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Expedition":{"affinity":{"Government":0.0,"Military":34.31,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":27.92,"Clan":0.0,"Cultural":0.0,"Outsider":87.62,"Elite":0.0,"Poor":0.0,"Academic":38.79},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Nomads":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":27.92,"Clan":34.31,"Cultural":38.79,"Outsider":91.91,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Diaspora":{"affinity":{"Government":0.0,"Military":0.0,"Religious":34.31,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":27.92,"Clan":0.0,"Cultural":74.17,"Outsider":94.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Confederation":{"affinity":{"Government":60.59,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":0.0,"Clan":52.77,"Cultural":83.14,"Outsider":57.57,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":50.0,"S":0.0}},"Assembly":{"affinity":{"Government":52.77,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":57.57,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":71.62,"Outsider":0.0,"Elite":0.0,"Poor":85.4,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Coalition":{"affinity":{"Government":43.85,"Military":0.0,"Religious":0.0,"Criminal":35.47,"Revolutionary":60.59,"Conspiracy":0.0,"Merchant":59.09,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":51.09,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":50.0,"S":0.0}},"Horde":{"affinity":{"Government":0.0,"Military":70.32,"Religious":0.0,"Criminal":67.66,"Revolutionary":54.4,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":57.57,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Mercenaries":{"affinity":{"Government":0.0,"Military":72.91,"Religious":0.0,"Criminal":69.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":34.31,"Clan":0.0,"Cultural":0.0,"Outsider":27.92,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Parliament":{"affinity":{"Government":77.86,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":34.31,"Conspiracy":0.0,"Merchant":0.0,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":79.65,"Poor":27.92,"Academic":0.0},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Senate":{"affinity":{"Government":75.42,"Military":0.0,"Religious":0.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":27.92,"Clan":0.0,"Cultural":0.0,"Outsider":0.0,"Elite":81.99,"Poor":0.0,"Academic":34.31},"grammar":{"M":100.0,"O":0.0,"P":0.0,"S":0.0}},"Association":{"affinity":{"Government":0.0,"Military":0.0,"Religious":0.0,"Criminal":30.61,"Revolutionary":0.0,"Conspiracy":0.0,"Merchant":72.91,"Clan":0.0,"Cultural":34.31,"Outsider":0.0,"Elite":0.0,"Poor":54.4,"Academic":66.29},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}},"Compact":{"affinity":{"Government":43.85,"Military":0.0,"Religious":0.0,"Criminal":42.88,"Revolutionary":38.79,"Conspiracy":56.0,"Merchant":34.31,"Clan":27.92,"Cultural":0.0,"Outsider":0.0,"Elite":0.0,"Poor":0.0,"Academic":0.0},"grammar":{"M":100.0,"O":50.0,"P":0.0,"S":0.0}}}}')
TYPES = BOOK1['types']
ANIMAL = BOOK1['animal']
BASE = BOOK1['base']
assert len(TYPES) == 13 and len(ANIMAL) == 56 and len(BASE) == 84

In [ ]:
def generate_slots(rng):
    slots = [{'id': f'F{i}', 'types': []} for i in range(1,8)]
    slots[0]['types'] = ['Government']
    slots[1]['types'] = ['Revolutionary']
    # Stage 2: two independent, distinct recipient slots.
    elite_index, poor_index = rng.sample(range(1,7),2)
    slots[elite_index]['types'].append('Elite')
    slots[poor_index]['types'].append('Poor')
    # Stage 3: only one Outsider, any F2..F7 that has room.
    eligible = [i for i in range(1,7) if len(slots[i]['types']) <= 1]
    slots[rng.choice(eligible)]['types'].append('Outsider')
    # Stage 4: fill empty slots in randomized order, with unique leftover Types.
    remaining = [t for t in TYPES if t not in {'Government','Revolutionary','Elite','Poor','Outsider'}]
    assert len(remaining)==8
    rng.shuffle(remaining)
    empty = [i for i,s in enumerate(slots) if not s['types']]
    rng.shuffle(empty)
    for i in empty:
        slots[i]['types'].append(remaining.pop())
    # Stage 5: give each remaining Type to one random single-Type slot.
    while remaining:
        open_slots = [i for i,s in enumerate(slots) if len(s['types']) == 1]
        slots[rng.choice(open_slots)]['types'].append(remaining.pop())
    validate_slots(slots)
    return slots

def validate_slots(slots):
    assert len(slots) == 7
    assert slots[0]['types'][0] == 'Government'
    assert slots[1]['types'][0] == 'Revolutionary'
    assert all(1 <= len(s['types']) <= 2 for s in slots)
    assert all('Government' not in s['types'] or not ({'Elite','Poor','Revolutionary','Outsider'} & set(s['types'])) for s in slots)
    loc={t:i for i,s in enumerate(slots) for t in s['types']}
    assert len(loc) == 13 and set(loc) == set(TYPES)
    assert loc['Elite'] != loc['Poor']
    assert all(s['id'] == f'F{i}' for i,s in enumerate(slots,1))

def naming_score(x,y,rng):
    # Average of (arithmetic mean, geometric mean, uniform random 0..100).
    return ((x+y)/2 + math.sqrt(x*y) + rng.uniform(0,100))/3

In [ ]:
def candidate_ranking(types, catalog, rng, item_type):
    results=[]
    for label, data in catalog.items():
        aff = data if item_type == 'Animal' else data['affinity']
        x=aff[types[0]]
        y=aff[types[1]] if len(types)==2 else x
        score=naming_score(x,y,rng)
        results.append({'label':label,'score':score,'x':x,'y':y})
    # Lexicographic stable tie-break, no extra random calls.
    results.sort(key=lambda z:(-z['score'],z['label']))
    return results

def choose_grammar(base, rng):
    weights=BASE[base]['grammar']
    labels=[g for g in ('M','O','P','S') if weights[g]>0]
    assert labels,base
    return rng.choices(labels,weights=[weights[g] for g in labels],k=1)[0]

# Singular/plural wording for Animal grammar. Book1 supplies labels/affinity only,
# so regular plurals are defaults, with common irregular spellings handled explicitly.
IRREGULAR={'Wolf':'Wolves','Fox':'Foxes','Sheep':'Sheep','Butterfly':'Butterflies',
           'Swallow':'Swallows','Octopus':'Octopuses','Phoenix':'Phoenixes'}
def plural(animal):
    if animal in IRREGULAR: return IRREGULAR[animal]
    if animal.endswith(('s','x','z','ch','sh')): return animal+'es'
    if animal.endswith('y') and animal[-2].lower() not in 'aeiou': return animal[:-1]+'ies'
    return animal+'s'

def faction_name(base, animal, grammar):
    if grammar=='M': return f'{animal} {base}'
    if grammar=='O': return f'{base} of the {animal}'
    if grammar=='P': return f'{base} of the {plural(animal)}'
    if grammar=='S': return f"{animal}'s {base}"
    raise ValueError(grammar)

def generate_world(seed=20261120, retain_rankings=False):
    rng=random.Random(seed)
    slots=generate_slots(rng)
    factions=slots[:6]  # F7 is discarded permanently before any naming occurs.
    used_base=set(); used_animal=set()
    for faction in factions:
        types=faction['types']
        base_rank=candidate_ranking(types, BASE, rng, 'Base')
        animal_rank=candidate_ranking(types, ANIMAL, rng, 'Animal')
        best_base=next(c for c in base_rank if c['label'] not in used_base)
        best_animal=next(c for c in animal_rank if c['label'] not in used_animal)
        used_base.add(best_base['label']); used_animal.add(best_animal['label'])
        grammar=choose_grammar(best_base['label'],rng)
        faction.update({
            'base': best_base['label'], 'base_score': round(best_base['score'],5),
            'base_affinities': [best_base['x'],best_base['y']],
            'animal': best_animal['label'], 'animal_score': round(best_animal['score'],5),
            'animal_affinities': [best_animal['x'],best_animal['y']],
            'grammar': grammar,
            'name': faction_name(best_base['label'],best_animal['label'],grammar)
        })
        if retain_rankings:
            faction['base_candidates']=base_rank
            faction['animal_candidates']=animal_rank
    assert len(factions)==6
    assert len({f['base'] for f in factions})==6
    assert len({f['animal'] for f in factions})==6
    assert len({f['name'] for f in factions})==6
    return {'version':VERSION,'seed':seed,'source':BOOK1['source'],
            'factions':factions}


## Zone Selection — locked v0.4.1

Generate 7–9 unique Zone Types; F1–F6 each selects one sequentially, then remaining 1–3 without Faction fit. Source: Book1.xlsx snapshot (36 Scale 4–5 Zone Types).

Faction base = AVG(AVG(Type1, Type2, Base), CUBERT(Type1 × Type2 × Base), Random[0,100]); extra Zone base = Random[0,100]. For single Type, use it twice.

Define SOFT(S)=AVG(AVG(S), MAX(S)), or 0 when S is empty. Candidate score = base + 0.75×SOFT(Functions under minimum) − SOFT(Functions at/over maximum AND affinity >=75) − SOFT(Functions over maximum AND affinity >=75) + Zone Type modifier. Bonus looks at all raw Function affinities; penalties only count Function affinities >=75. Over-maximum Function is penalized twice. After selection increment ALL Functions with affinity >=75 and recompute.

Modifiers are additive points, not percentage multipliers: Caravan Camp +7; Estate +2; Military Base −3; Mining Settlement +4; Rebel Camp −4; Cultural District −1; Entertainment District −4; Market District −2; Royal Quarter −7. All 27 other Zone Types = 0.

No fixed F1 Government/F2 Revolutionary bias is added in this locked configuration; related ideas were discussed but not calibrated as part of the approved 5–40% dataset. Greedy selection, unique Types, and soft quotas remain as before.


In [ ]:
ZONE_DATA_JSON = '{"source":"Book1.xlsx (2026-11-20, Zone Scale 4-5 rows)","function_quotas":{"Authority":{"min":1,"max":1},"Defense":{"min":1,"max":2},"Religion":{"min":0,"max":1},"Knowledge":{"min":0,"max":1},"Commerce":{"min":1,"max":2},"Production":{"min":0,"max":2},"Logistics":{"min":0,"max":2},"Resources":{"min":0,"max":2},"Residence":{"min":1,"max":2},"Social":{"min":1,"max":2},"Covert":{"min":0,"max":1},"Migration":{"min":0,"max":1},"Care":{"min":0,"max":1},"Exploration":{"min":0,"max":2}},"zones":{"Caravan Camp":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":50.0,"Production":0.0,"Logistics":100.0,"Resources":0.0,"Residence":75.0,"Social":0.0,"Covert":0.0,"Migration":100.0,"Care":0.0,"Exploration":25.0},"type":{"Government":10.0,"Military":15.0,"Religious":10.0,"Criminal":50.0,"Revolutionary":20.0,"Conspiracy":10.0,"Merchant":85.0,"Clan":65.0,"Cultural":10.0,"Outsider":90.0,"Elite":20.0,"Poor":60.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":20.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":20.0,"Army":20.0,"Militia":20.0,"Vanguard":20.0,"Watch":20.0,"Corps":20.0,"Wardens":20.0,"Rangers":45.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":65.0,"Fellowship":65.0,"Chapter":50.0,"Disciples":15.0,"Cult":15.0,"Guild":45.0,"Company":45.0,"Consortium":70.0,"Syndicate":70.0,"Cartel":70.0,"Cooperative":45.0,"Corporation":70.0,"Chamber":70.0,"Trust":45.0,"Clan":65.0,"House":20.0,"Dynasty":40.0,"Family":65.0,"Tribe":75.0,"People":35.0,"Community":40.0,"Nation":35.0,"Kindred":65.0,"Lineage":40.0,"Resistance":20.0,"Movement":20.0,"Front":20.0,"Union":40.0,"Collective":40.0,"Alliance":55.0,"League":55.0,"Pact":55.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":50.0,"Circle":50.0,"Cabal":50.0,"Conclave":50.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":100.0,"Fleet":75.0,"Band":100.0,"Enclave":55.0,"Commune":70.0,"Network":75.0,"Expedition":100.0,"Nomads":100.0,"Diaspora":75.0,"Confederation":45.0,"Assembly":45.0,"Coalition":45.0,"Horde":45.0,"Mercenaries":45.0,"Parliament":45.0,"Senate":45.0,"Association":45.0,"Compact":70.0}},"Cavern":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":25.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":25.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":100.0},"type":{"Government":10.0,"Military":10.0,"Religious":10.0,"Criminal":65.0,"Revolutionary":45.0,"Conspiracy":70.0,"Merchant":10.0,"Clan":35.0,"Cultural":10.0,"Outsider":80.0,"Elite":10.0,"Poor":70.0,"Academic":60.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":35.0,"Vanguard":35.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":45.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":50.0,"Brotherhood":35.0,"Fellowship":35.0,"Chapter":15.0,"Disciples":50.0,"Cult":70.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":65.0,"Cartel":65.0,"Cooperative":40.0,"Corporation":15.0,"Chamber":15.0,"Trust":15.0,"Clan":35.0,"House":10.0,"Dynasty":35.0,"Family":35.0,"Tribe":45.0,"People":35.0,"Community":40.0,"Nation":30.0,"Kindred":35.0,"Lineage":35.0,"Resistance":60.0,"Movement":35.0,"Front":60.0,"Union":40.0,"Collective":40.0,"Alliance":35.0,"League":35.0,"Pact":40.0,"Liberation":60.0,"Academy":40.0,"Institute":40.0,"School":15.0,"Society":40.0,"Circle":40.0,"Cabal":70.0,"Conclave":70.0,"Scholars":40.0,"Collegium":40.0,"Keepers":40.0,"Caravan":45.0,"Fleet":45.0,"Band":70.0,"Enclave":5.0,"Commune":40.0,"Network":70.0,"Expedition":90.0,"Nomads":45.0,"Diaspora":45.0,"Confederation":40.0,"Assembly":40.0,"Coalition":40.0,"Horde":45.0,"Mercenaries":45.0,"Parliament":45.0,"Senate":45.0,"Association":45.0,"Compact":40.0}},"Citadel":{"scale":4,"function":{"Authority":95.0,"Defense":100.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":25.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":85.0,"Military":90.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":20.0,"Conspiracy":10.0,"Merchant":10.0,"Clan":65.0,"Cultural":10.0,"Outsider":10.0,"Elite":60.0,"Poor":10.0,"Academic":10.0},"base":{"Empire":95.0,"Kingdom":95.0,"Republic":70.0,"Dominion":95.0,"Council":95.0,"Court":70.0,"Commonwealth":70.0,"Directorate":95.0,"Ministry":95.0,"Authority":95.0,"Legion":100.0,"Guard":100.0,"Army":100.0,"Militia":75.0,"Vanguard":75.0,"Watch":100.0,"Corps":100.0,"Wardens":95.0,"Rangers":70.0,"Order":100.0,"Covenant":75.0,"Faith":75.0,"Sect":15.0,"Brotherhood":35.0,"Fellowship":35.0,"Chapter":15.0,"Disciples":15.0,"Cult":15.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":45.0,"Trust":65.0,"Clan":65.0,"House":15.0,"Dynasty":65.0,"Family":35.0,"Tribe":60.0,"People":35.0,"Community":35.0,"Nation":30.0,"Kindred":65.0,"Lineage":60.0,"Resistance":55.0,"Movement":55.0,"Front":55.0,"Union":20.0,"Collective":20.0,"Alliance":55.0,"League":55.0,"Pact":55.0,"Liberation":55.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":15.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":50.0,"Caravan":15.0,"Fleet":45.0,"Band":50.0,"Enclave":5.0,"Commune":15.0,"Network":15.0,"Expedition":50.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":70.0,"Assembly":70.0,"Coalition":70.0,"Horde":70.0,"Mercenaries":70.0,"Parliament":95.0,"Senate":95.0,"Association":15.0,"Compact":45.0}},"Enclave":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":100.0,"Social":0.0,"Covert":0.0,"Migration":75.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":45.0,"Criminal":10.0,"Revolutionary":55.0,"Conspiracy":65.0,"Merchant":10.0,"Clan":95.0,"Cultural":10.0,"Outsider":85.0,"Elite":20.0,"Poor":25.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":20.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":35.0,"Vanguard":35.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":45.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":95.0,"Fellowship":95.0,"Chapter":85.0,"Disciples":35.0,"Cult":40.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":30.0,"Corporation":15.0,"Chamber":15.0,"Trust":20.0,"Clan":95.0,"House":75.0,"Dynasty":45.0,"Family":95.0,"Tribe":95.0,"People":45.0,"Community":45.0,"Nation":40.0,"Kindred":95.0,"Lineage":45.0,"Resistance":40.0,"Movement":40.0,"Front":40.0,"Union":40.0,"Collective":40.0,"Alliance":90.0,"League":90.0,"Pact":90.0,"Liberation":35.0,"Academy":20.0,"Institute":20.0,"School":15.0,"Society":75.0,"Circle":90.0,"Cabal":55.0,"Conclave":45.0,"Scholars":15.0,"Collegium":15.0,"Keepers":35.0,"Caravan":70.0,"Fleet":45.0,"Band":70.0,"Enclave":100.0,"Commune":95.0,"Network":45.0,"Expedition":70.0,"Nomads":95.0,"Diaspora":95.0,"Confederation":45.0,"Assembly":45.0,"Coalition":45.0,"Horde":45.0,"Mercenaries":45.0,"Parliament":45.0,"Senate":45.0,"Association":45.0,"Compact":95.0}},"Estate":{"scale":4,"function":{"Authority":55.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":50.0,"Residence":100.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":15.0,"Military":10.0,"Religious":45.0,"Criminal":10.0,"Revolutionary":15.0,"Conspiracy":10.0,"Merchant":50.0,"Clan":45.0,"Cultural":10.0,"Outsider":20.0,"Elite":80.0,"Poor":25.0,"Academic":10.0},"base":{"Empire":55.0,"Kingdom":80.0,"Republic":20.0,"Dominion":55.0,"Council":20.0,"Court":70.0,"Commonwealth":20.0,"Directorate":20.0,"Ministry":50.0,"Authority":55.0,"Legion":20.0,"Guard":20.0,"Army":20.0,"Militia":20.0,"Vanguard":20.0,"Watch":20.0,"Corps":15.0,"Wardens":20.0,"Rangers":20.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":45.0,"Fellowship":45.0,"Chapter":35.0,"Disciples":35.0,"Cult":35.0,"Guild":35.0,"Company":35.0,"Consortium":35.0,"Syndicate":35.0,"Cartel":35.0,"Cooperative":35.0,"Corporation":35.0,"Chamber":35.0,"Trust":70.0,"Clan":45.0,"House":75.0,"Dynasty":95.0,"Family":95.0,"Tribe":45.0,"People":45.0,"Community":45.0,"Nation":40.0,"Kindred":45.0,"Lineage":95.0,"Resistance":20.0,"Movement":20.0,"Front":20.0,"Union":30.0,"Collective":30.0,"Alliance":20.0,"League":20.0,"Pact":20.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":15.0,"Society":50.0,"Circle":50.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":35.0,"Caravan":35.0,"Fleet":35.0,"Band":20.0,"Enclave":55.0,"Commune":15.0,"Network":20.0,"Expedition":20.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":45.0,"Assembly":20.0,"Coalition":15.0,"Horde":20.0,"Mercenaries":35.0,"Parliament":55.0,"Senate":70.0,"Association":35.0,"Compact":35.0}},"Village":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":25.0,"Production":0.0,"Logistics":0.0,"Resources":75.0,"Residence":100.0,"Social":75.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":50.0,"Criminal":10.0,"Revolutionary":15.0,"Conspiracy":10.0,"Merchant":75.0,"Clan":70.0,"Cultural":25.0,"Outsider":55.0,"Elite":20.0,"Poor":85.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":50.0,"Dominion":15.0,"Council":15.0,"Court":20.0,"Commonwealth":50.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":20.0,"Vanguard":20.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":35.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":45.0,"Fellowship":45.0,"Chapter":35.0,"Disciples":35.0,"Cult":35.0,"Guild":45.0,"Company":45.0,"Consortium":45.0,"Syndicate":70.0,"Cartel":70.0,"Cooperative":95.0,"Corporation":45.0,"Chamber":40.0,"Trust":40.0,"Clan":70.0,"House":90.0,"Dynasty":45.0,"Family":90.0,"Tribe":90.0,"People":95.0,"Community":95.0,"Nation":90.0,"Kindred":70.0,"Lineage":45.0,"Resistance":20.0,"Movement":55.0,"Front":20.0,"Union":70.0,"Collective":70.0,"Alliance":20.0,"League":20.0,"Pact":20.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":20.0,"Society":30.0,"Circle":20.0,"Cabal":50.0,"Conclave":50.0,"Scholars":15.0,"Collegium":15.0,"Keepers":35.0,"Caravan":45.0,"Fleet":40.0,"Band":35.0,"Enclave":60.0,"Commune":85.0,"Network":35.0,"Expedition":65.0,"Nomads":40.0,"Diaspora":40.0,"Confederation":45.0,"Assembly":35.0,"Coalition":35.0,"Horde":35.0,"Mercenaries":40.0,"Parliament":35.0,"Senate":35.0,"Association":40.0,"Compact":40.0}},"Hunting Grounds":{"scale":4,"function":{"Authority":25.0,"Defense":0.0,"Religion":0.0,"Knowledge":25.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":100.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":50.0},"type":{"Government":10.0,"Military":30.0,"Religious":35.0,"Criminal":25.0,"Revolutionary":40.0,"Conspiracy":10.0,"Merchant":45.0,"Clan":50.0,"Cultural":10.0,"Outsider":65.0,"Elite":10.0,"Poor":75.0,"Academic":15.0},"base":{"Empire":30.0,"Kingdom":30.0,"Republic":30.0,"Dominion":25.0,"Council":25.0,"Court":25.0,"Commonwealth":30.0,"Directorate":25.0,"Ministry":25.0,"Authority":25.0,"Legion":40.0,"Guard":35.0,"Army":40.0,"Militia":50.0,"Vanguard":40.0,"Watch":35.0,"Corps":40.0,"Wardens":35.0,"Rangers":90.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":50.0,"Fellowship":45.0,"Chapter":35.0,"Disciples":35.0,"Cult":35.0,"Guild":35.0,"Company":40.0,"Consortium":35.0,"Syndicate":40.0,"Cartel":40.0,"Cooperative":70.0,"Corporation":35.0,"Chamber":35.0,"Trust":35.0,"Clan":50.0,"House":0.0,"Dynasty":45.0,"Family":50.0,"Tribe":50.0,"People":65.0,"Community":70.0,"Nation":60.0,"Kindred":50.0,"Lineage":45.0,"Resistance":50.0,"Movement":50.0,"Front":50.0,"Union":60.0,"Collective":60.0,"Alliance":50.0,"League":50.0,"Pact":50.0,"Liberation":50.0,"Academy":30.0,"Institute":30.0,"School":0.0,"Society":30.0,"Circle":30.0,"Cabal":30.0,"Conclave":30.0,"Scholars":30.0,"Collegium":30.0,"Keepers":35.0,"Caravan":55.0,"Fleet":45.0,"Band":55.0,"Enclave":5.0,"Commune":65.0,"Network":40.0,"Expedition":55.0,"Nomads":55.0,"Diaspora":55.0,"Confederation":40.0,"Assembly":45.0,"Coalition":40.0,"Horde":55.0,"Mercenaries":55.0,"Parliament":40.0,"Senate":40.0,"Association":40.0,"Compact":40.0}},"Military Base":{"scale":4,"function":{"Authority":70.0,"Defense":100.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":75.0,"Resources":0.0,"Residence":50.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":80.0,"Military":100.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":65.0,"Conspiracy":10.0,"Merchant":20.0,"Clan":65.0,"Cultural":10.0,"Outsider":35.0,"Elite":15.0,"Poor":30.0,"Academic":10.0},"base":{"Empire":70.0,"Kingdom":70.0,"Republic":50.0,"Dominion":70.0,"Council":70.0,"Court":50.0,"Commonwealth":50.0,"Directorate":70.0,"Ministry":70.0,"Authority":70.0,"Legion":100.0,"Guard":90.0,"Army":100.0,"Militia":90.0,"Vanguard":90.0,"Watch":90.0,"Corps":100.0,"Wardens":90.0,"Rangers":100.0,"Order":75.0,"Covenant":50.0,"Faith":50.0,"Sect":25.0,"Brotherhood":40.0,"Fellowship":40.0,"Chapter":25.0,"Disciples":25.0,"Cult":25.0,"Guild":20.0,"Company":20.0,"Consortium":20.0,"Syndicate":35.0,"Cartel":35.0,"Cooperative":35.0,"Corporation":20.0,"Chamber":45.0,"Trust":30.0,"Clan":65.0,"House":20.0,"Dynasty":40.0,"Family":45.0,"Tribe":65.0,"People":40.0,"Community":40.0,"Nation":35.0,"Kindred":65.0,"Lineage":40.0,"Resistance":65.0,"Movement":65.0,"Front":90.0,"Union":45.0,"Collective":45.0,"Alliance":65.0,"League":65.0,"Pact":65.0,"Liberation":65.0,"Academy":30.0,"Institute":30.0,"School":15.0,"Society":25.0,"Circle":30.0,"Cabal":30.0,"Conclave":30.0,"Scholars":25.0,"Collegium":25.0,"Keepers":45.0,"Caravan":30.0,"Fleet":55.0,"Band":55.0,"Enclave":30.0,"Commune":40.0,"Network":30.0,"Expedition":60.0,"Nomads":35.0,"Diaspora":35.0,"Confederation":70.0,"Assembly":70.0,"Coalition":70.0,"Horde":90.0,"Mercenaries":90.0,"Parliament":70.0,"Senate":70.0,"Association":30.0,"Compact":45.0}},"Mining Settlement":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":25.0,"Logistics":0.0,"Resources":100.0,"Residence":100.0,"Social":0.0,"Covert":0.0,"Migration":25.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":45.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":15.0,"Conspiracy":10.0,"Merchant":75.0,"Clan":60.0,"Cultural":10.0,"Outsider":60.0,"Elite":20.0,"Poor":85.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":50.0,"Dominion":15.0,"Council":15.0,"Court":20.0,"Commonwealth":50.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":35.0,"Guard":35.0,"Army":35.0,"Militia":35.0,"Vanguard":35.0,"Watch":35.0,"Corps":35.0,"Wardens":35.0,"Rangers":40.0,"Order":35.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":30.0,"Fellowship":30.0,"Chapter":15.0,"Disciples":15.0,"Cult":15.0,"Guild":45.0,"Company":45.0,"Consortium":45.0,"Syndicate":70.0,"Cartel":70.0,"Cooperative":70.0,"Corporation":45.0,"Chamber":40.0,"Trust":40.0,"Clan":60.0,"House":80.0,"Dynasty":30.0,"Family":55.0,"Tribe":65.0,"People":55.0,"Community":70.0,"Nation":50.0,"Kindred":55.0,"Lineage":30.0,"Resistance":20.0,"Movement":55.0,"Front":20.0,"Union":70.0,"Collective":70.0,"Alliance":20.0,"League":20.0,"Pact":20.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":15.0,"Society":15.0,"Circle":20.0,"Cabal":50.0,"Conclave":50.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":45.0,"Fleet":40.0,"Band":40.0,"Enclave":60.0,"Commune":60.0,"Network":40.0,"Expedition":90.0,"Nomads":40.0,"Diaspora":40.0,"Confederation":35.0,"Assembly":35.0,"Coalition":35.0,"Horde":40.0,"Mercenaries":40.0,"Parliament":40.0,"Senate":40.0,"Association":40.0,"Compact":40.0}},"Monastery":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":100.0,"Knowledge":75.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":75.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":25.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":90.0,"Criminal":10.0,"Revolutionary":15.0,"Conspiracy":55.0,"Merchant":10.0,"Clan":40.0,"Cultural":55.0,"Outsider":55.0,"Elite":20.0,"Poor":20.0,"Academic":85.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":50.0,"Court":20.0,"Commonwealth":15.0,"Directorate":50.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":20.0,"Vanguard":20.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":35.0,"Order":100.0,"Covenant":100.0,"Faith":100.0,"Sect":100.0,"Brotherhood":100.0,"Fellowship":95.0,"Chapter":95.0,"Disciples":100.0,"Cult":100.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":20.0,"Corporation":15.0,"Chamber":15.0,"Trust":20.0,"Clan":40.0,"House":30.0,"Dynasty":40.0,"Family":40.0,"Tribe":40.0,"People":35.0,"Community":40.0,"Nation":35.0,"Kindred":40.0,"Lineage":40.0,"Resistance":20.0,"Movement":20.0,"Front":20.0,"Union":20.0,"Collective":20.0,"Alliance":20.0,"League":20.0,"Pact":35.0,"Liberation":20.0,"Academy":95.0,"Institute":95.0,"School":45.0,"Society":70.0,"Circle":70.0,"Cabal":45.0,"Conclave":45.0,"Scholars":95.0,"Collegium":95.0,"Keepers":95.0,"Caravan":40.0,"Fleet":35.0,"Band":35.0,"Enclave":30.0,"Commune":35.0,"Network":35.0,"Expedition":45.0,"Nomads":40.0,"Diaspora":40.0,"Confederation":35.0,"Assembly":35.0,"Coalition":35.0,"Horde":35.0,"Mercenaries":35.0,"Parliament":35.0,"Senate":35.0,"Association":65.0,"Compact":35.0}},"Necropolis":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":75.0,"Knowledge":25.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":0.0,"Covert":25.0,"Migration":0.0,"Care":0.0,"Exploration":75.0},"type":{"Government":10.0,"Military":10.0,"Religious":85.0,"Criminal":15.0,"Revolutionary":10.0,"Conspiracy":70.0,"Merchant":10.0,"Clan":40.0,"Cultural":55.0,"Outsider":20.0,"Elite":10.0,"Poor":10.0,"Academic":55.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":15.0,"Vanguard":15.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":20.0,"Order":70.0,"Covenant":70.0,"Faith":70.0,"Sect":70.0,"Brotherhood":70.0,"Fellowship":70.0,"Chapter":70.0,"Disciples":70.0,"Cult":70.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":15.0,"Trust":15.0,"Clan":40.0,"House":10.0,"Dynasty":40.0,"Family":40.0,"Tribe":40.0,"People":40.0,"Community":40.0,"Nation":35.0,"Kindred":40.0,"Lineage":40.0,"Resistance":20.0,"Movement":20.0,"Front":15.0,"Union":20.0,"Collective":20.0,"Alliance":15.0,"League":15.0,"Pact":40.0,"Liberation":15.0,"Academy":65.0,"Institute":65.0,"School":25.0,"Society":35.0,"Circle":40.0,"Cabal":40.0,"Conclave":40.0,"Scholars":65.0,"Collegium":65.0,"Keepers":70.0,"Caravan":20.0,"Fleet":20.0,"Band":20.0,"Enclave":5.0,"Commune":15.0,"Network":40.0,"Expedition":35.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":40.0,"Assembly":20.0,"Coalition":15.0,"Horde":20.0,"Mercenaries":20.0,"Parliament":20.0,"Senate":20.0,"Association":35.0,"Compact":15.0}},"Rebel Camp":{"scale":4,"function":{"Authority":50.0,"Defense":100.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":75.0,"Social":0.0,"Covert":75.0,"Migration":25.0,"Care":0.0,"Exploration":25.0},"type":{"Government":0.0,"Military":85.0,"Religious":10.0,"Criminal":30.0,"Revolutionary":100.0,"Conspiracy":65.0,"Merchant":10.0,"Clan":55.0,"Cultural":10.0,"Outsider":55.0,"Elite":20.0,"Poor":60.0,"Academic":10.0},"base":{"Empire":50.0,"Kingdom":50.0,"Republic":15.0,"Dominion":50.0,"Council":50.0,"Court":20.0,"Commonwealth":15.0,"Directorate":50.0,"Ministry":50.0,"Authority":50.0,"Legion":70.0,"Guard":70.0,"Army":70.0,"Militia":100.0,"Vanguard":100.0,"Watch":70.0,"Corps":70.0,"Wardens":70.0,"Rangers":70.0,"Order":70.0,"Covenant":50.0,"Faith":50.0,"Sect":15.0,"Brotherhood":20.0,"Fellowship":20.0,"Chapter":15.0,"Disciples":15.0,"Cult":40.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":40.0,"Cartel":40.0,"Cooperative":40.0,"Corporation":15.0,"Chamber":15.0,"Trust":20.0,"Clan":55.0,"House":25.0,"Dynasty":20.0,"Family":20.0,"Tribe":65.0,"People":20.0,"Community":40.0,"Nation":15.0,"Kindred":55.0,"Lineage":20.0,"Resistance":100.0,"Movement":100.0,"Front":100.0,"Union":50.0,"Collective":50.0,"Alliance":75.0,"League":75.0,"Pact":75.0,"Liberation":100.0,"Academy":20.0,"Institute":20.0,"School":15.0,"Society":15.0,"Circle":40.0,"Cabal":40.0,"Conclave":40.0,"Scholars":15.0,"Collegium":15.0,"Keepers":50.0,"Caravan":40.0,"Fleet":45.0,"Band":90.0,"Enclave":35.0,"Commune":45.0,"Network":40.0,"Expedition":65.0,"Nomads":40.0,"Diaspora":40.0,"Confederation":60.0,"Assembly":60.0,"Coalition":60.0,"Horde":95.0,"Mercenaries":95.0,"Parliament":65.0,"Senate":65.0,"Association":35.0,"Compact":35.0}},"Refugee Camp":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":100.0,"Social":25.0,"Covert":0.0,"Migration":100.0,"Care":75.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":50.0,"Criminal":10.0,"Revolutionary":60.0,"Conspiracy":10.0,"Merchant":10.0,"Clan":65.0,"Cultural":15.0,"Outsider":100.0,"Elite":20.0,"Poor":90.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":20.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":40.0,"Vanguard":40.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":50.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":65.0,"Fellowship":65.0,"Chapter":60.0,"Disciples":35.0,"Cult":35.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":45.0,"Corporation":15.0,"Chamber":15.0,"Trust":20.0,"Clan":65.0,"House":75.0,"Dynasty":40.0,"Family":65.0,"Tribe":75.0,"People":65.0,"Community":70.0,"Nation":60.0,"Kindred":65.0,"Lineage":40.0,"Resistance":40.0,"Movement":65.0,"Front":40.0,"Union":75.0,"Collective":75.0,"Alliance":65.0,"League":65.0,"Pact":65.0,"Liberation":90.0,"Academy":20.0,"Institute":20.0,"School":20.0,"Society":55.0,"Circle":50.0,"Cabal":50.0,"Conclave":50.0,"Scholars":15.0,"Collegium":15.0,"Keepers":35.0,"Caravan":75.0,"Fleet":50.0,"Band":75.0,"Enclave":70.0,"Commune":70.0,"Network":50.0,"Expedition":75.0,"Nomads":75.0,"Diaspora":100.0,"Confederation":50.0,"Assembly":45.0,"Coalition":45.0,"Horde":50.0,"Mercenaries":50.0,"Parliament":50.0,"Senate":50.0,"Association":50.0,"Compact":75.0}},"Slum":{"scale":4,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":100.0,"Social":50.0,"Covert":25.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":5.0,"Military":10.0,"Religious":10.0,"Criminal":55.0,"Revolutionary":75.0,"Conspiracy":15.0,"Merchant":10.0,"Clan":30.0,"Cultural":20.0,"Outsider":55.0,"Elite":0.0,"Poor":100.0,"Academic":10.0},"base":{"Empire":15.0,"Kingdom":15.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":45.0,"Vanguard":45.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":35.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":30.0,"Fellowship":30.0,"Chapter":15.0,"Disciples":15.0,"Cult":20.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":40.0,"Cartel":40.0,"Cooperative":50.0,"Corporation":15.0,"Chamber":15.0,"Trust":15.0,"Clan":30.0,"House":85.0,"Dynasty":10.0,"Family":30.0,"Tribe":35.0,"People":55.0,"Community":90.0,"Nation":50.0,"Kindred":30.0,"Lineage":30.0,"Resistance":90.0,"Movement":70.0,"Front":45.0,"Union":90.0,"Collective":75.0,"Alliance":40.0,"League":40.0,"Pact":40.0,"Liberation":90.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":20.0,"Circle":20.0,"Cabal":20.0,"Conclave":20.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":40.0,"Fleet":35.0,"Band":35.0,"Enclave":55.0,"Commune":40.0,"Network":35.0,"Expedition":35.0,"Nomads":40.0,"Diaspora":65.0,"Confederation":35.0,"Assembly":35.0,"Coalition":35.0,"Horde":35.0,"Mercenaries":35.0,"Parliament":35.0,"Senate":35.0,"Association":35.0,"Compact":35.0}},"University":{"scale":4,"function":{"Authority":45.0,"Defense":0.0,"Religion":0.0,"Knowledge":100.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":50.0,"Social":50.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":50.0,"Military":20.0,"Religious":40.0,"Criminal":15.0,"Revolutionary":25.0,"Conspiracy":35.0,"Merchant":20.0,"Clan":40.0,"Cultural":50.0,"Outsider":20.0,"Elite":35.0,"Poor":25.0,"Academic":90.0},"base":{"Empire":45.0,"Kingdom":45.0,"Republic":45.0,"Dominion":45.0,"Council":60.0,"Court":45.0,"Commonwealth":45.0,"Directorate":65.0,"Ministry":45.0,"Authority":45.0,"Legion":30.0,"Guard":40.0,"Army":30.0,"Militia":35.0,"Vanguard":30.0,"Watch":40.0,"Corps":30.0,"Wardens":40.0,"Rangers":30.0,"Order":65.0,"Covenant":65.0,"Faith":65.0,"Sect":60.0,"Brotherhood":60.0,"Fellowship":60.0,"Chapter":65.0,"Disciples":60.0,"Cult":65.0,"Guild":35.0,"Company":35.0,"Consortium":35.0,"Syndicate":30.0,"Cartel":30.0,"Cooperative":40.0,"Corporation":35.0,"Chamber":45.0,"Trust":35.0,"Clan":40.0,"House":50.0,"Dynasty":40.0,"Family":35.0,"Tribe":40.0,"People":35.0,"Community":40.0,"Nation":35.0,"Kindred":40.0,"Lineage":40.0,"Resistance":35.0,"Movement":35.0,"Front":35.0,"Union":35.0,"Collective":35.0,"Alliance":30.0,"League":30.0,"Pact":35.0,"Liberation":35.0,"Academy":100.0,"Institute":100.0,"School":80.0,"Society":90.0,"Circle":80.0,"Cabal":60.0,"Conclave":60.0,"Scholars":100.0,"Collegium":100.0,"Keepers":75.0,"Caravan":30.0,"Fleet":30.0,"Band":30.0,"Enclave":45.0,"Commune":30.0,"Network":35.0,"Expedition":50.0,"Nomads":30.0,"Diaspora":30.0,"Confederation":40.0,"Assembly":40.0,"Coalition":35.0,"Horde":30.0,"Mercenaries":30.0,"Parliament":45.0,"Senate":45.0,"Association":60.0,"Compact":35.0}},"Academic District":{"scale":5,"function":{"Authority":35.0,"Defense":0.0,"Religion":0.0,"Knowledge":100.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":50.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":50.0,"Military":10.0,"Religious":70.0,"Criminal":10.0,"Revolutionary":10.0,"Conspiracy":75.0,"Merchant":10.0,"Clan":20.0,"Cultural":25.0,"Outsider":10.0,"Elite":10.0,"Poor":10.0,"Academic":100.0},"base":{"Empire":35.0,"Kingdom":35.0,"Republic":35.0,"Dominion":35.0,"Council":85.0,"Court":35.0,"Commonwealth":35.0,"Directorate":85.0,"Ministry":35.0,"Authority":35.0,"Legion":20.0,"Guard":35.0,"Army":20.0,"Militia":15.0,"Vanguard":15.0,"Watch":35.0,"Corps":15.0,"Wardens":35.0,"Rangers":15.0,"Order":70.0,"Covenant":70.0,"Faith":70.0,"Sect":70.0,"Brotherhood":65.0,"Fellowship":65.0,"Chapter":65.0,"Disciples":65.0,"Cult":70.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":35.0,"Trust":15.0,"Clan":20.0,"House":15.0,"Dynasty":20.0,"Family":20.0,"Tribe":20.0,"People":15.0,"Community":20.0,"Nation":15.0,"Kindred":20.0,"Lineage":20.0,"Resistance":20.0,"Movement":20.0,"Front":15.0,"Union":20.0,"Collective":20.0,"Alliance":15.0,"League":15.0,"Pact":40.0,"Liberation":15.0,"Academy":100.0,"Institute":100.0,"School":100.0,"Society":100.0,"Circle":100.0,"Cabal":50.0,"Conclave":50.0,"Scholars":100.0,"Collegium":100.0,"Keepers":75.0,"Caravan":15.0,"Fleet":15.0,"Band":15.0,"Enclave":5.0,"Commune":15.0,"Network":40.0,"Expedition":50.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":35.0,"Assembly":35.0,"Coalition":35.0,"Horde":15.0,"Mercenaries":15.0,"Parliament":35.0,"Senate":35.0,"Association":80.0,"Compact":35.0}},"Government District":{"scale":5,"function":{"Authority":100.0,"Defense":25.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":25.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":100.0,"Military":55.0,"Religious":10.0,"Criminal":0.0,"Revolutionary":0.0,"Conspiracy":15.0,"Merchant":10.0,"Clan":20.0,"Cultural":15.0,"Outsider":10.0,"Elite":65.0,"Poor":10.0,"Academic":45.0},"base":{"Empire":75.0,"Kingdom":75.0,"Republic":100.0,"Dominion":75.0,"Council":100.0,"Court":100.0,"Commonwealth":100.0,"Directorate":100.0,"Ministry":100.0,"Authority":100.0,"Legion":65.0,"Guard":75.0,"Army":65.0,"Militia":35.0,"Vanguard":35.0,"Watch":75.0,"Corps":65.0,"Wardens":75.0,"Rangers":35.0,"Order":65.0,"Covenant":50.0,"Faith":50.0,"Sect":15.0,"Brotherhood":15.0,"Fellowship":15.0,"Chapter":15.0,"Disciples":15.0,"Cult":20.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":50.0,"Trust":90.0,"Clan":20.0,"House":10.0,"Dynasty":65.0,"Family":15.0,"Tribe":15.0,"People":15.0,"Community":15.0,"Nation":10.0,"Kindred":15.0,"Lineage":50.0,"Resistance":15.0,"Movement":15.0,"Front":15.0,"Union":20.0,"Collective":20.0,"Alliance":15.0,"League":15.0,"Pact":20.0,"Liberation":15.0,"Academy":35.0,"Institute":35.0,"School":10.0,"Society":35.0,"Circle":35.0,"Cabal":35.0,"Conclave":35.0,"Scholars":35.0,"Collegium":35.0,"Keepers":35.0,"Caravan":15.0,"Fleet":35.0,"Band":15.0,"Enclave":0.0,"Commune":15.0,"Network":20.0,"Expedition":35.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":100.0,"Assembly":100.0,"Coalition":100.0,"Horde":35.0,"Mercenaries":35.0,"Parliament":100.0,"Senate":100.0,"Association":20.0,"Compact":45.0}},"Artisan Quarter":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":100.0,"Production":100.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":25.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":10.0,"Criminal":15.0,"Revolutionary":45.0,"Conspiracy":10.0,"Merchant":95.0,"Clan":35.0,"Cultural":70.0,"Outsider":10.0,"Elite":20.0,"Poor":65.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":50.0,"Dominion":15.0,"Council":15.0,"Court":20.0,"Commonwealth":50.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":35.0,"Vanguard":35.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":15.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":85.0,"Fellowship":85.0,"Chapter":75.0,"Disciples":15.0,"Cult":15.0,"Guild":100.0,"Company":100.0,"Consortium":75.0,"Syndicate":75.0,"Cartel":75.0,"Cooperative":100.0,"Corporation":75.0,"Chamber":75.0,"Trust":75.0,"Clan":35.0,"House":15.0,"Dynasty":35.0,"Family":35.0,"Tribe":35.0,"People":35.0,"Community":40.0,"Nation":30.0,"Kindred":35.0,"Lineage":35.0,"Resistance":35.0,"Movement":35.0,"Front":35.0,"Union":90.0,"Collective":90.0,"Alliance":60.0,"League":60.0,"Pact":60.0,"Liberation":35.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":40.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":75.0,"Fleet":75.0,"Band":20.0,"Enclave":5.0,"Commune":30.0,"Network":75.0,"Expedition":15.0,"Nomads":50.0,"Diaspora":20.0,"Confederation":60.0,"Assembly":50.0,"Coalition":50.0,"Horde":15.0,"Mercenaries":45.0,"Parliament":15.0,"Senate":20.0,"Association":100.0,"Compact":70.0}},"Cultural District":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":25.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":100.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":50.0,"Criminal":15.0,"Revolutionary":45.0,"Conspiracy":10.0,"Merchant":10.0,"Clan":40.0,"Cultural":100.0,"Outsider":10.0,"Elite":65.0,"Poor":10.0,"Academic":70.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":40.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":35.0,"Vanguard":35.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":15.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":35.0,"Fellowship":35.0,"Chapter":35.0,"Disciples":35.0,"Cult":35.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":15.0,"Trust":40.0,"Clan":40.0,"House":20.0,"Dynasty":40.0,"Family":40.0,"Tribe":35.0,"People":35.0,"Community":35.0,"Nation":30.0,"Kindred":35.0,"Lineage":35.0,"Resistance":35.0,"Movement":85.0,"Front":35.0,"Union":35.0,"Collective":35.0,"Alliance":35.0,"League":35.0,"Pact":35.0,"Liberation":35.0,"Academy":40.0,"Institute":40.0,"School":30.0,"Society":50.0,"Circle":40.0,"Cabal":40.0,"Conclave":40.0,"Scholars":40.0,"Collegium":40.0,"Keepers":40.0,"Caravan":15.0,"Fleet":15.0,"Band":20.0,"Enclave":5.0,"Commune":30.0,"Network":20.0,"Expedition":40.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":35.0,"Assembly":15.0,"Coalition":15.0,"Horde":15.0,"Mercenaries":15.0,"Parliament":15.0,"Senate":40.0,"Association":50.0,"Compact":15.0}},"Entertainment District":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":75.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":100.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":15.0,"Criminal":75.0,"Revolutionary":10.0,"Conspiracy":10.0,"Merchant":85.0,"Clan":20.0,"Cultural":90.0,"Outsider":45.0,"Elite":55.0,"Poor":45.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":50.0,"Dominion":15.0,"Council":15.0,"Court":40.0,"Commonwealth":50.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":15.0,"Vanguard":15.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":35.0,"Order":20.0,"Covenant":20.0,"Faith":20.0,"Sect":20.0,"Brotherhood":55.0,"Fellowship":55.0,"Chapter":50.0,"Disciples":20.0,"Cult":20.0,"Guild":70.0,"Company":70.0,"Consortium":70.0,"Syndicate":70.0,"Cartel":70.0,"Cooperative":70.0,"Corporation":70.0,"Chamber":70.0,"Trust":70.0,"Clan":20.0,"House":20.0,"Dynasty":40.0,"Family":20.0,"Tribe":35.0,"People":20.0,"Community":35.0,"Nation":15.0,"Kindred":20.0,"Lineage":20.0,"Resistance":20.0,"Movement":50.0,"Front":15.0,"Union":60.0,"Collective":60.0,"Alliance":50.0,"League":50.0,"Pact":50.0,"Liberation":15.0,"Academy":20.0,"Institute":20.0,"School":20.0,"Society":45.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":70.0,"Fleet":70.0,"Band":45.0,"Enclave":0.0,"Commune":30.0,"Network":70.0,"Expedition":35.0,"Nomads":60.0,"Diaspora":35.0,"Confederation":60.0,"Assembly":60.0,"Coalition":60.0,"Horde":35.0,"Mercenaries":45.0,"Parliament":35.0,"Senate":35.0,"Association":70.0,"Compact":70.0}},"Farmland":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":25.0,"Logistics":0.0,"Resources":100.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":45.0,"Criminal":10.0,"Revolutionary":10.0,"Conspiracy":10.0,"Merchant":75.0,"Clan":45.0,"Cultural":10.0,"Outsider":45.0,"Elite":10.0,"Poor":75.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":15.0,"Vanguard":15.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":35.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":40.0,"Fellowship":40.0,"Chapter":35.0,"Disciples":35.0,"Cult":35.0,"Guild":45.0,"Company":45.0,"Consortium":45.0,"Syndicate":45.0,"Cartel":45.0,"Cooperative":70.0,"Corporation":45.0,"Chamber":40.0,"Trust":40.0,"Clan":45.0,"House":10.0,"Dynasty":45.0,"Family":45.0,"Tribe":40.0,"People":65.0,"Community":70.0,"Nation":65.0,"Kindred":45.0,"Lineage":45.0,"Resistance":20.0,"Movement":20.0,"Front":15.0,"Union":45.0,"Collective":45.0,"Alliance":15.0,"League":15.0,"Pact":15.0,"Liberation":15.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":15.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":35.0,"Caravan":45.0,"Fleet":40.0,"Band":35.0,"Enclave":5.0,"Commune":55.0,"Network":35.0,"Expedition":35.0,"Nomads":35.0,"Diaspora":35.0,"Confederation":40.0,"Assembly":30.0,"Coalition":30.0,"Horde":35.0,"Mercenaries":40.0,"Parliament":35.0,"Senate":35.0,"Association":40.0,"Compact":40.0}},"Foreign Quarter":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":50.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":100.0,"Social":50.0,"Covert":0.0,"Migration":100.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":10.0,"Criminal":15.0,"Revolutionary":20.0,"Conspiracy":10.0,"Merchant":65.0,"Clan":65.0,"Cultural":60.0,"Outsider":100.0,"Elite":25.0,"Poor":65.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":30.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":20.0,"Vanguard":20.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":50.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":65.0,"Fellowship":65.0,"Chapter":50.0,"Disciples":15.0,"Cult":15.0,"Guild":40.0,"Company":40.0,"Consortium":40.0,"Syndicate":40.0,"Cartel":40.0,"Cooperative":40.0,"Corporation":40.0,"Chamber":40.0,"Trust":40.0,"Clan":65.0,"House":80.0,"Dynasty":40.0,"Family":65.0,"Tribe":75.0,"People":40.0,"Community":40.0,"Nation":35.0,"Kindred":65.0,"Lineage":40.0,"Resistance":20.0,"Movement":20.0,"Front":20.0,"Union":40.0,"Collective":40.0,"Alliance":55.0,"League":55.0,"Pact":55.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":65.0,"Circle":50.0,"Cabal":50.0,"Conclave":50.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":100.0,"Fleet":50.0,"Band":75.0,"Enclave":100.0,"Commune":70.0,"Network":50.0,"Expedition":75.0,"Nomads":75.0,"Diaspora":100.0,"Confederation":50.0,"Assembly":45.0,"Coalition":45.0,"Horde":50.0,"Mercenaries":50.0,"Parliament":50.0,"Senate":50.0,"Association":50.0,"Compact":75.0}},"Wilderness":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":50.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":75.0},"type":{"Government":5.0,"Military":10.0,"Religious":45.0,"Criminal":20.0,"Revolutionary":45.0,"Conspiracy":10.0,"Merchant":10.0,"Clan":40.0,"Cultural":10.0,"Outsider":90.0,"Elite":10.0,"Poor":30.0,"Academic":15.0},"base":{"Empire":15.0,"Kingdom":15.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":35.0,"Vanguard":35.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":90.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":35.0,"Brotherhood":40.0,"Fellowship":40.0,"Chapter":35.0,"Disciples":35.0,"Cult":35.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":35.0,"Cartel":35.0,"Cooperative":70.0,"Corporation":15.0,"Chamber":15.0,"Trust":15.0,"Clan":40.0,"House":10.0,"Dynasty":40.0,"Family":40.0,"Tribe":90.0,"People":65.0,"Community":65.0,"Nation":60.0,"Kindred":40.0,"Lineage":40.0,"Resistance":35.0,"Movement":35.0,"Front":35.0,"Union":40.0,"Collective":40.0,"Alliance":35.0,"League":35.0,"Pact":35.0,"Liberation":35.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":20.0,"Circle":20.0,"Cabal":20.0,"Conclave":20.0,"Scholars":20.0,"Collegium":20.0,"Keepers":35.0,"Caravan":45.0,"Fleet":45.0,"Band":45.0,"Enclave":5.0,"Commune":70.0,"Network":45.0,"Expedition":90.0,"Nomads":90.0,"Diaspora":45.0,"Confederation":45.0,"Assembly":45.0,"Coalition":45.0,"Horde":90.0,"Mercenaries":45.0,"Parliament":45.0,"Senate":45.0,"Association":45.0,"Compact":45.0}},"Harbor":{"scale":5,"function":{"Authority":35.0,"Defense":25.0,"Religion":0.0,"Knowledge":0.0,"Commerce":100.0,"Production":0.0,"Logistics":100.0,"Resources":0.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":75.0,"Care":0.0,"Exploration":0.0},"type":{"Government":50.0,"Military":55.0,"Religious":10.0,"Criminal":55.0,"Revolutionary":15.0,"Conspiracy":10.0,"Merchant":95.0,"Clan":20.0,"Cultural":10.0,"Outsider":85.0,"Elite":20.0,"Poor":15.0,"Academic":10.0},"base":{"Empire":35.0,"Kingdom":35.0,"Republic":65.0,"Dominion":35.0,"Council":35.0,"Court":35.0,"Commonwealth":60.0,"Directorate":35.0,"Ministry":35.0,"Authority":35.0,"Legion":40.0,"Guard":40.0,"Army":40.0,"Militia":35.0,"Vanguard":35.0,"Watch":35.0,"Corps":35.0,"Wardens":35.0,"Rangers":45.0,"Order":40.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":50.0,"Fellowship":50.0,"Chapter":50.0,"Disciples":15.0,"Cult":15.0,"Guild":75.0,"Company":75.0,"Consortium":100.0,"Syndicate":75.0,"Cartel":90.0,"Cooperative":75.0,"Corporation":100.0,"Chamber":100.0,"Trust":75.0,"Clan":20.0,"House":10.0,"Dynasty":20.0,"Family":15.0,"Tribe":45.0,"People":15.0,"Community":20.0,"Nation":10.0,"Kindred":15.0,"Lineage":15.0,"Resistance":20.0,"Movement":20.0,"Front":20.0,"Union":55.0,"Collective":55.0,"Alliance":55.0,"League":50.0,"Pact":50.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":15.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":75.0,"Fleet":100.0,"Band":70.0,"Enclave":35.0,"Commune":40.0,"Network":70.0,"Expedition":70.0,"Nomads":70.0,"Diaspora":45.0,"Confederation":70.0,"Assembly":70.0,"Coalition":70.0,"Horde":45.0,"Mercenaries":45.0,"Parliament":45.0,"Senate":45.0,"Association":75.0,"Compact":70.0}},"Industrial District":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":100.0,"Logistics":50.0,"Resources":0.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":50.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":45.0,"Conspiracy":10.0,"Merchant":80.0,"Clan":20.0,"Cultural":45.0,"Outsider":10.0,"Elite":10.0,"Poor":75.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":35.0,"Guard":35.0,"Army":35.0,"Militia":35.0,"Vanguard":35.0,"Watch":35.0,"Corps":35.0,"Wardens":35.0,"Rangers":35.0,"Order":35.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":50.0,"Fellowship":50.0,"Chapter":50.0,"Disciples":15.0,"Cult":15.0,"Guild":95.0,"Company":95.0,"Consortium":95.0,"Syndicate":45.0,"Cartel":45.0,"Cooperative":95.0,"Corporation":95.0,"Chamber":95.0,"Trust":45.0,"Clan":20.0,"House":10.0,"Dynasty":20.0,"Family":15.0,"Tribe":15.0,"People":15.0,"Community":40.0,"Nation":10.0,"Kindred":15.0,"Lineage":15.0,"Resistance":35.0,"Movement":35.0,"Front":35.0,"Union":95.0,"Collective":95.0,"Alliance":35.0,"League":35.0,"Pact":35.0,"Liberation":35.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":35.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":45.0,"Fleet":45.0,"Band":15.0,"Enclave":0.0,"Commune":30.0,"Network":50.0,"Expedition":15.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":15.0,"Assembly":15.0,"Coalition":15.0,"Horde":35.0,"Mercenaries":45.0,"Parliament":15.0,"Senate":15.0,"Association":70.0,"Compact":40.0}},"Market District":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":100.0,"Production":0.0,"Logistics":25.0,"Resources":0.0,"Residence":0.0,"Social":75.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":10.0,"Criminal":75.0,"Revolutionary":10.0,"Conspiracy":10.0,"Merchant":100.0,"Clan":20.0,"Cultural":25.0,"Outsider":45.0,"Elite":55.0,"Poor":10.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":75.0,"Dominion":15.0,"Council":15.0,"Court":40.0,"Commonwealth":75.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":15.0,"Vanguard":15.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":35.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":50.0,"Fellowship":50.0,"Chapter":50.0,"Disciples":15.0,"Cult":15.0,"Guild":100.0,"Company":100.0,"Consortium":100.0,"Syndicate":75.0,"Cartel":75.0,"Cooperative":75.0,"Corporation":100.0,"Chamber":100.0,"Trust":100.0,"Clan":20.0,"House":15.0,"Dynasty":40.0,"Family":20.0,"Tribe":35.0,"People":15.0,"Community":20.0,"Nation":15.0,"Kindred":20.0,"Lineage":20.0,"Resistance":20.0,"Movement":20.0,"Front":15.0,"Union":50.0,"Collective":50.0,"Alliance":75.0,"League":75.0,"Pact":75.0,"Liberation":15.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":30.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":100.0,"Fleet":75.0,"Band":45.0,"Enclave":0.0,"Commune":30.0,"Network":70.0,"Expedition":35.0,"Nomads":85.0,"Diaspora":35.0,"Confederation":85.0,"Assembly":85.0,"Coalition":85.0,"Horde":35.0,"Mercenaries":50.0,"Parliament":35.0,"Senate":35.0,"Association":100.0,"Compact":100.0}},"Military District":{"scale":5,"function":{"Authority":70.0,"Defense":100.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":50.0,"Resources":0.0,"Residence":25.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":75.0,"Military":90.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":80.0,"Conspiracy":10.0,"Merchant":15.0,"Clan":65.0,"Cultural":10.0,"Outsider":10.0,"Elite":10.0,"Poor":10.0,"Academic":10.0},"base":{"Empire":70.0,"Kingdom":70.0,"Republic":45.0,"Dominion":70.0,"Council":70.0,"Court":45.0,"Commonwealth":45.0,"Directorate":70.0,"Ministry":70.0,"Authority":70.0,"Legion":100.0,"Guard":100.0,"Army":100.0,"Militia":75.0,"Vanguard":75.0,"Watch":100.0,"Corps":100.0,"Wardens":95.0,"Rangers":95.0,"Order":75.0,"Covenant":50.0,"Faith":50.0,"Sect":15.0,"Brotherhood":35.0,"Fellowship":35.0,"Chapter":15.0,"Disciples":15.0,"Cult":15.0,"Guild":20.0,"Company":20.0,"Consortium":20.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":20.0,"Corporation":20.0,"Chamber":40.0,"Trust":20.0,"Clan":65.0,"House":15.0,"Dynasty":35.0,"Family":35.0,"Tribe":60.0,"People":35.0,"Community":35.0,"Nation":30.0,"Kindred":65.0,"Lineage":35.0,"Resistance":70.0,"Movement":70.0,"Front":70.0,"Union":45.0,"Collective":45.0,"Alliance":70.0,"League":70.0,"Pact":70.0,"Liberation":70.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":15.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":50.0,"Caravan":20.0,"Fleet":45.0,"Band":50.0,"Enclave":5.0,"Commune":40.0,"Network":15.0,"Expedition":75.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":65.0,"Assembly":65.0,"Coalition":65.0,"Horde":70.0,"Mercenaries":70.0,"Parliament":70.0,"Senate":70.0,"Association":20.0,"Compact":40.0}},"Mining Region":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":25.0,"Logistics":0.0,"Resources":100.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":25.0},"type":{"Government":10.0,"Military":45.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":10.0,"Conspiracy":10.0,"Merchant":75.0,"Clan":20.0,"Cultural":10.0,"Outsider":50.0,"Elite":10.0,"Poor":85.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":35.0,"Guard":35.0,"Army":35.0,"Militia":35.0,"Vanguard":35.0,"Watch":35.0,"Corps":35.0,"Wardens":35.0,"Rangers":35.0,"Order":35.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":20.0,"Fellowship":20.0,"Chapter":15.0,"Disciples":15.0,"Cult":15.0,"Guild":45.0,"Company":45.0,"Consortium":45.0,"Syndicate":45.0,"Cartel":45.0,"Cooperative":45.0,"Corporation":45.0,"Chamber":40.0,"Trust":40.0,"Clan":20.0,"House":10.0,"Dynasty":20.0,"Family":20.0,"Tribe":35.0,"People":20.0,"Community":45.0,"Nation":15.0,"Kindred":20.0,"Lineage":20.0,"Resistance":20.0,"Movement":20.0,"Front":15.0,"Union":45.0,"Collective":45.0,"Alliance":15.0,"League":15.0,"Pact":15.0,"Liberation":15.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":15.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":45.0,"Fleet":40.0,"Band":35.0,"Enclave":5.0,"Commune":30.0,"Network":35.0,"Expedition":65.0,"Nomads":35.0,"Diaspora":35.0,"Confederation":35.0,"Assembly":35.0,"Coalition":35.0,"Horde":35.0,"Mercenaries":40.0,"Parliament":35.0,"Senate":35.0,"Association":40.0,"Compact":40.0}},"Noble Quarter":{"scale":5,"function":{"Authority":65.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":100.0,"Social":25.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":60.0,"Military":10.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":15.0,"Conspiracy":70.0,"Merchant":10.0,"Clan":40.0,"Cultural":15.0,"Outsider":20.0,"Elite":100.0,"Poor":0.0,"Academic":10.0},"base":{"Empire":65.0,"Kingdom":65.0,"Republic":40.0,"Dominion":65.0,"Council":40.0,"Court":75.0,"Commonwealth":40.0,"Directorate":40.0,"Ministry":65.0,"Authority":65.0,"Legion":20.0,"Guard":40.0,"Army":20.0,"Militia":20.0,"Vanguard":20.0,"Watch":40.0,"Corps":15.0,"Wardens":40.0,"Rangers":20.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":40.0,"Fellowship":40.0,"Chapter":15.0,"Disciples":15.0,"Cult":40.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":40.0,"Trust":75.0,"Clan":40.0,"House":70.0,"Dynasty":90.0,"Family":65.0,"Tribe":40.0,"People":40.0,"Community":40.0,"Nation":35.0,"Kindred":40.0,"Lineage":65.0,"Resistance":20.0,"Movement":20.0,"Front":20.0,"Union":20.0,"Collective":20.0,"Alliance":20.0,"League":20.0,"Pact":40.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":55.0,"Circle":70.0,"Cabal":40.0,"Conclave":40.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":20.0,"Fleet":20.0,"Band":20.0,"Enclave":55.0,"Commune":15.0,"Network":40.0,"Expedition":20.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":40.0,"Assembly":35.0,"Coalition":35.0,"Horde":20.0,"Mercenaries":20.0,"Parliament":65.0,"Senate":75.0,"Association":20.0,"Compact":35.0}},"Royal Quarter":{"scale":5,"function":{"Authority":100.0,"Defense":25.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":75.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":95.0,"Military":15.0,"Religious":45.0,"Criminal":10.0,"Revolutionary":15.0,"Conspiracy":50.0,"Merchant":10.0,"Clan":45.0,"Cultural":45.0,"Outsider":15.0,"Elite":100.0,"Poor":0.0,"Academic":10.0},"base":{"Empire":100.0,"Kingdom":100.0,"Republic":75.0,"Dominion":100.0,"Council":75.0,"Court":100.0,"Commonwealth":75.0,"Directorate":75.0,"Ministry":100.0,"Authority":100.0,"Legion":55.0,"Guard":75.0,"Army":55.0,"Militia":20.0,"Vanguard":20.0,"Watch":75.0,"Corps":55.0,"Wardens":75.0,"Rangers":20.0,"Order":60.0,"Covenant":60.0,"Faith":60.0,"Sect":35.0,"Brotherhood":45.0,"Fellowship":45.0,"Chapter":35.0,"Disciples":35.0,"Cult":35.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":50.0,"Trust":75.0,"Clan":45.0,"House":55.0,"Dynasty":100.0,"Family":70.0,"Tribe":45.0,"People":40.0,"Community":45.0,"Nation":40.0,"Kindred":45.0,"Lineage":95.0,"Resistance":20.0,"Movement":20.0,"Front":20.0,"Union":20.0,"Collective":20.0,"Alliance":20.0,"League":20.0,"Pact":35.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":60.0,"Circle":65.0,"Cabal":35.0,"Conclave":35.0,"Scholars":15.0,"Collegium":15.0,"Keepers":35.0,"Caravan":20.0,"Fleet":20.0,"Band":20.0,"Enclave":45.0,"Commune":15.0,"Network":35.0,"Expedition":20.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":75.0,"Assembly":70.0,"Coalition":70.0,"Horde":15.0,"Mercenaries":20.0,"Parliament":100.0,"Senate":100.0,"Association":35.0,"Compact":45.0}},"Residential District":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":100.0,"Social":50.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":10.0,"Criminal":10.0,"Revolutionary":15.0,"Conspiracy":10.0,"Merchant":10.0,"Clan":95.0,"Cultural":75.0,"Outsider":55.0,"Elite":60.0,"Poor":80.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":75.0,"Dominion":15.0,"Council":15.0,"Court":40.0,"Commonwealth":75.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":20.0,"Vanguard":20.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":35.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":45.0,"Fellowship":45.0,"Chapter":15.0,"Disciples":15.0,"Cult":15.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":50.0,"Cartel":50.0,"Cooperative":70.0,"Corporation":15.0,"Chamber":15.0,"Trust":40.0,"Clan":95.0,"House":100.0,"Dynasty":45.0,"Family":95.0,"Tribe":95.0,"People":95.0,"Community":95.0,"Nation":90.0,"Kindred":95.0,"Lineage":45.0,"Resistance":20.0,"Movement":55.0,"Front":20.0,"Union":70.0,"Collective":70.0,"Alliance":20.0,"League":20.0,"Pact":20.0,"Liberation":20.0,"Academy":20.0,"Institute":20.0,"School":35.0,"Society":40.0,"Circle":20.0,"Cabal":50.0,"Conclave":50.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":40.0,"Fleet":35.0,"Band":35.0,"Enclave":60.0,"Commune":85.0,"Network":35.0,"Expedition":65.0,"Nomads":40.0,"Diaspora":40.0,"Confederation":45.0,"Assembly":35.0,"Coalition":35.0,"Horde":35.0,"Mercenaries":35.0,"Parliament":35.0,"Senate":40.0,"Association":40.0,"Compact":35.0}},"Ruined Quarter":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":25.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":25.0,"Social":0.0,"Covert":25.0,"Migration":0.0,"Care":0.0,"Exploration":100.0},"type":{"Government":10.0,"Military":10.0,"Religious":45.0,"Criminal":55.0,"Revolutionary":50.0,"Conspiracy":70.0,"Merchant":10.0,"Clan":20.0,"Cultural":10.0,"Outsider":65.0,"Elite":10.0,"Poor":50.0,"Academic":75.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":35.0,"Vanguard":35.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":40.0,"Order":35.0,"Covenant":35.0,"Faith":35.0,"Sect":60.0,"Brotherhood":35.0,"Fellowship":35.0,"Chapter":35.0,"Disciples":60.0,"Cult":70.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":65.0,"Cartel":65.0,"Cooperative":35.0,"Corporation":15.0,"Chamber":15.0,"Trust":15.0,"Clan":20.0,"House":70.0,"Dynasty":20.0,"Family":15.0,"Tribe":40.0,"People":15.0,"Community":35.0,"Nation":10.0,"Kindred":15.0,"Lineage":15.0,"Resistance":90.0,"Movement":35.0,"Front":65.0,"Union":35.0,"Collective":35.0,"Alliance":35.0,"League":35.0,"Pact":40.0,"Liberation":65.0,"Academy":45.0,"Institute":45.0,"School":25.0,"Society":40.0,"Circle":45.0,"Cabal":70.0,"Conclave":70.0,"Scholars":45.0,"Collegium":45.0,"Keepers":40.0,"Caravan":40.0,"Fleet":40.0,"Band":65.0,"Enclave":10.0,"Commune":35.0,"Network":70.0,"Expedition":45.0,"Nomads":40.0,"Diaspora":40.0,"Confederation":40.0,"Assembly":40.0,"Coalition":35.0,"Horde":40.0,"Mercenaries":40.0,"Parliament":40.0,"Senate":40.0,"Association":40.0,"Compact":35.0}},"Temple District":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":100.0,"Knowledge":0.0,"Commerce":0.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":75.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":10.0,"Religious":100.0,"Criminal":10.0,"Revolutionary":10.0,"Conspiracy":45.0,"Merchant":10.0,"Clan":40.0,"Cultural":65.0,"Outsider":10.0,"Elite":10.0,"Poor":10.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":15.0,"Vanguard":15.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":15.0,"Order":100.0,"Covenant":100.0,"Faith":100.0,"Sect":100.0,"Brotherhood":75.0,"Fellowship":75.0,"Chapter":75.0,"Disciples":100.0,"Cult":100.0,"Guild":20.0,"Company":20.0,"Consortium":15.0,"Syndicate":20.0,"Cartel":20.0,"Cooperative":15.0,"Corporation":15.0,"Chamber":15.0,"Trust":15.0,"Clan":40.0,"House":15.0,"Dynasty":40.0,"Family":40.0,"Tribe":35.0,"People":35.0,"Community":35.0,"Nation":30.0,"Kindred":35.0,"Lineage":35.0,"Resistance":20.0,"Movement":20.0,"Front":15.0,"Union":20.0,"Collective":20.0,"Alliance":15.0,"League":15.0,"Pact":35.0,"Liberation":15.0,"Academy":50.0,"Institute":50.0,"School":40.0,"Society":40.0,"Circle":35.0,"Cabal":35.0,"Conclave":35.0,"Scholars":50.0,"Collegium":50.0,"Keepers":100.0,"Caravan":15.0,"Fleet":15.0,"Band":15.0,"Enclave":5.0,"Commune":15.0,"Network":35.0,"Expedition":15.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":35.0,"Assembly":15.0,"Coalition":15.0,"Horde":15.0,"Mercenaries":15.0,"Parliament":15.0,"Senate":15.0,"Association":40.0,"Compact":15.0}},"Underground City":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":50.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":100.0,"Social":0.0,"Covert":50.0,"Migration":25.0,"Care":0.0,"Exploration":50.0},"type":{"Government":10.0,"Military":10.0,"Religious":10.0,"Criminal":80.0,"Revolutionary":55.0,"Conspiracy":75.0,"Merchant":60.0,"Clan":55.0,"Cultural":10.0,"Outsider":85.0,"Elite":25.0,"Poor":80.0,"Academic":15.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":50.0,"Dominion":15.0,"Council":15.0,"Court":30.0,"Commonwealth":50.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":20.0,"Guard":15.0,"Army":20.0,"Militia":35.0,"Vanguard":35.0,"Watch":15.0,"Corps":15.0,"Wardens":15.0,"Rangers":45.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":50.0,"Brotherhood":30.0,"Fellowship":30.0,"Chapter":15.0,"Disciples":50.0,"Cult":70.0,"Guild":40.0,"Company":40.0,"Consortium":40.0,"Syndicate":95.0,"Cartel":95.0,"Cooperative":70.0,"Corporation":40.0,"Chamber":40.0,"Trust":40.0,"Clan":55.0,"House":70.0,"Dynasty":30.0,"Family":55.0,"Tribe":70.0,"People":55.0,"Community":70.0,"Nation":50.0,"Kindred":55.0,"Lineage":30.0,"Resistance":65.0,"Movement":65.0,"Front":65.0,"Union":70.0,"Collective":70.0,"Alliance":35.0,"League":35.0,"Pact":40.0,"Liberation":65.0,"Academy":20.0,"Institute":20.0,"School":15.0,"Society":20.0,"Circle":45.0,"Cabal":40.0,"Conclave":40.0,"Scholars":20.0,"Collegium":20.0,"Keepers":15.0,"Caravan":45.0,"Fleet":45.0,"Band":70.0,"Enclave":60.0,"Commune":70.0,"Network":70.0,"Expedition":70.0,"Nomads":45.0,"Diaspora":45.0,"Confederation":45.0,"Assembly":45.0,"Coalition":45.0,"Horde":45.0,"Mercenaries":45.0,"Parliament":45.0,"Senate":45.0,"Association":45.0,"Compact":45.0}},"Underworld":{"scale":5,"function":{"Authority":20.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":75.0,"Production":0.0,"Logistics":0.0,"Resources":0.0,"Residence":0.0,"Social":25.0,"Covert":100.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":15.0,"Military":10.0,"Religious":10.0,"Criminal":100.0,"Revolutionary":50.0,"Conspiracy":90.0,"Merchant":40.0,"Clan":20.0,"Cultural":15.0,"Outsider":10.0,"Elite":55.0,"Poor":45.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":20.0,"Dominion":20.0,"Council":20.0,"Court":40.0,"Commonwealth":20.0,"Directorate":20.0,"Ministry":15.0,"Authority":20.0,"Legion":20.0,"Guard":20.0,"Army":20.0,"Militia":45.0,"Vanguard":45.0,"Watch":20.0,"Corps":15.0,"Wardens":20.0,"Rangers":15.0,"Order":20.0,"Covenant":20.0,"Faith":15.0,"Sect":75.0,"Brotherhood":15.0,"Fellowship":15.0,"Chapter":15.0,"Disciples":75.0,"Cult":45.0,"Guild":35.0,"Company":35.0,"Consortium":35.0,"Syndicate":100.0,"Cartel":100.0,"Cooperative":35.0,"Corporation":35.0,"Chamber":35.0,"Trust":35.0,"Clan":20.0,"House":10.0,"Dynasty":40.0,"Family":15.0,"Tribe":15.0,"People":15.0,"Community":35.0,"Nation":10.0,"Kindred":15.0,"Lineage":15.0,"Resistance":95.0,"Movement":45.0,"Front":95.0,"Union":45.0,"Collective":45.0,"Alliance":45.0,"League":45.0,"Pact":45.0,"Liberation":95.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":20.0,"Circle":45.0,"Cabal":100.0,"Conclave":45.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":35.0,"Fleet":35.0,"Band":100.0,"Enclave":5.0,"Commune":40.0,"Network":100.0,"Expedition":15.0,"Nomads":20.0,"Diaspora":20.0,"Confederation":15.0,"Assembly":15.0,"Coalition":15.0,"Horde":15.0,"Mercenaries":30.0,"Parliament":20.0,"Senate":35.0,"Association":30.0,"Compact":30.0}},"Warehouse District":{"scale":5,"function":{"Authority":15.0,"Defense":0.0,"Religion":0.0,"Knowledge":0.0,"Commerce":25.0,"Production":25.0,"Logistics":100.0,"Resources":0.0,"Residence":0.0,"Social":0.0,"Covert":0.0,"Migration":0.0,"Care":0.0,"Exploration":0.0},"type":{"Government":10.0,"Military":75.0,"Religious":10.0,"Criminal":50.0,"Revolutionary":10.0,"Conspiracy":10.0,"Merchant":90.0,"Clan":20.0,"Cultural":10.0,"Outsider":65.0,"Elite":10.0,"Poor":10.0,"Academic":10.0},"base":{"Empire":20.0,"Kingdom":20.0,"Republic":15.0,"Dominion":15.0,"Council":15.0,"Court":15.0,"Commonwealth":15.0,"Directorate":15.0,"Ministry":15.0,"Authority":15.0,"Legion":45.0,"Guard":45.0,"Army":45.0,"Militia":45.0,"Vanguard":45.0,"Watch":40.0,"Corps":45.0,"Wardens":40.0,"Rangers":40.0,"Order":45.0,"Covenant":20.0,"Faith":15.0,"Sect":15.0,"Brotherhood":15.0,"Fellowship":15.0,"Chapter":15.0,"Disciples":15.0,"Cult":15.0,"Guild":45.0,"Company":45.0,"Consortium":75.0,"Syndicate":90.0,"Cartel":75.0,"Cooperative":45.0,"Corporation":75.0,"Chamber":75.0,"Trust":45.0,"Clan":20.0,"House":10.0,"Dynasty":20.0,"Family":15.0,"Tribe":40.0,"People":15.0,"Community":15.0,"Nation":10.0,"Kindred":15.0,"Lineage":15.0,"Resistance":20.0,"Movement":20.0,"Front":15.0,"Union":20.0,"Collective":20.0,"Alliance":15.0,"League":15.0,"Pact":15.0,"Liberation":15.0,"Academy":20.0,"Institute":20.0,"School":10.0,"Society":15.0,"Circle":20.0,"Cabal":20.0,"Conclave":15.0,"Scholars":15.0,"Collegium":15.0,"Keepers":15.0,"Caravan":75.0,"Fleet":100.0,"Band":65.0,"Enclave":0.0,"Commune":35.0,"Network":65.0,"Expedition":65.0,"Nomads":65.0,"Diaspora":40.0,"Confederation":40.0,"Assembly":40.0,"Coalition":35.0,"Horde":40.0,"Mercenaries":45.0,"Parliament":40.0,"Senate":40.0,"Association":45.0,"Compact":70.0}}}}'
# Book1.xlsx Zone source snapshot: exact Scale 4–5 places, 14 Function minima/maxima,
# and Place→Faction Type / Base affinities. No Place Priority and no proximity.
ZONE_DATA = json.loads(ZONE_DATA_JSON)
ZONE_CANDIDATES = ZONE_DATA['zones']
FUNCTION_QUOTAS = ZONE_DATA['function_quotas']
assert len(ZONE_CANDIDATES)==36 and len(FUNCTION_QUOTAS)==14

ZONE_BONUS_WEIGHT = 0.75
ZONE_PENALTY_WEIGHT = 1.0
ZONE_FUNCTION_THRESHOLD = 75
# Locked calibration: 10,000 independent holdout seeds (14,900–24,899),
# all 36 Zone Types appeared between 5% and 40%. Additive SCORE POINTS,
# not per-world quotas, not affinity-percent multipliers.
ZONE_SCORE_MODIFIERS = {'Caravan Camp': 7, 'Estate': 2, 'Military Base': -3, 'Mining Settlement': 4, 'Rebel Camp': -4, 'Cultural District': -1, 'Entertainment District': -4, 'Market District': -2, 'Royal Quarter': -7}
assert set(ZONE_SCORE_MODIFIERS).issubset(ZONE_CANDIDATES)

def _zone_functions(name):
    return {k for k,v in ZONE_CANDIDATES[name]['function'].items()
            if v >= ZONE_FUNCTION_THRESHOLD}

def _soft_max(vals):
    vals=list(vals)
    return (max(vals)+sum(vals)/len(vals))/2 if vals else 0.0

def _zone_coverage_score(place, counts):
    # Bonus under minimum uses ALL raw affinities, including 0/25/50.
    # Penalties only apply if this Zone would increment that Function (>=75).
    # Once the existing count exceeds maximum, it appears in both penalties.
    relevant=ZONE_CANDIDATES[place]['function']
    under=_soft_max(v for k,v in relevant.items()
                    if counts[k] < FUNCTION_QUOTAS[k]['min'])
    at_or_over=_soft_max(v for k,v in relevant.items()
                         if v >= ZONE_FUNCTION_THRESHOLD and counts[k] >= FUNCTION_QUOTAS[k]['max'])
    over=_soft_max(v for k,v in relevant.items()
                   if v >= ZONE_FUNCTION_THRESHOLD and counts[k] > FUNCTION_QUOTAS[k]['max'])
    return under,at_or_over,over

def _faction_zone_score(place, faction, rand, counts):
    aff=ZONE_CANDIDATES[place]
    x=aff['type'][faction['types'][0]]
    y=aff['type'][faction['types'][1]] if len(faction['types'])==2 else x
    b=aff['base'][faction['base']]
    arithmetic=(x+y+b)/3
    geometric=(x*y*b)**(1/3) if x*y*b else 0.0
    under,at,over=_zone_coverage_score(place,counts)
    base_score=(arithmetic+geometric+rand)/3
    modifier=ZONE_SCORE_MODIFIERS.get(place,0)
    return base_score+ZONE_BONUS_WEIGHT*under-ZONE_PENALTY_WEIGHT*(at+over)+modifier,{'arithmetic':arithmetic,'geometric':geometric,
                  'random':rand,'under_min':under,'at_or_over_max':at,'over_max':over,'modifier':modifier}

def _extra_zone_score(place,rand,counts):
    under,at,over=_zone_coverage_score(place,counts)
    modifier=ZONE_SCORE_MODIFIERS.get(place,0)
    return rand+ZONE_BONUS_WEIGHT*under-ZONE_PENALTY_WEIGHT*(at+over)+modifier,{'random':rand,'under_min':under,
                               'at_or_over_max':at,'over_max':over,'modifier':modifier}

def generate_zones(world, seed=None, desired_count=None):
    # Separate deterministic RNG; avoids perturbing existing Faction naming RNG.
    rng=random.Random(world['seed'] if seed is None else seed)
    if desired_count is None:
        desired_count=rng.randint(7,9)
    if desired_count not in (7,8,9): raise ValueError('Zone count must be 7–9')
    counts={k:0 for k in FUNCTION_QUOTAS}
    zones=[]; used=set()
    for i in range(desired_count):
        faction=world['factions'][i] if i<6 else None
        candidates=[]
        for place in ZONE_CANDIDATES:
            rand=rng.uniform(0,100)
            if place in used:continue
            if faction is not None:
                score,parts=_faction_zone_score(place,faction,rand,counts)
            else:
                score,parts=_extra_zone_score(place,rand,counts)
            candidates.append((score,place,parts))
        score,place,parts=max(candidates,key=lambda t:(t[0],t[1]))
        coverage=sorted(_zone_functions(place))
        for k in coverage:counts[k]+=1
        used.add(place)
        zones.append({'sequence':i+1,'place':place,'scale':ZONE_CANDIDATES[place]['scale'],
                      'selected_by':faction['id'] if faction else None,
                      'faction_name':faction['name'] if faction else None,
                      'score':round(score,5),'components':{k:round(v,5) for k,v in parts.items()},
                      'covered_functions':coverage})
    missing={k:max(0,q['min']-counts[k]) for k,q in FUNCTION_QUOTAS.items()}
    excess={k:max(0,counts[k]-q['max']) for k,q in FUNCTION_QUOTAS.items()}
    return {'target':desired_count,'zones':zones,'function_counts':counts,
            'missing_minimum':{k:v for k,v in missing.items() if v},
            'excess_maximum':{k:v for k,v in excess.items() if v}}

def generate_world_with_zones(seed=20261120,desired_count=None):
    world=generate_world(seed)
    world['zone_generation']=generate_zones(world,seed=seed+1000003,desired_count=desired_count)
    return world

def zone_qa(n=1000):
    from collections import Counter
    counts=Counter();miss=Counter();over=Counter();places=Counter();compositions=set()
    examples=[]
    for seed in range(n):
        w=generate_world_with_zones(seed)
        z=w['zone_generation'];zs=z['zones']
        assert len(zs) in (7,8,9)
        assert len({x['place'] for x in zs})==len(zs)
        assert [x['selected_by'] for x in zs[:6]]==[f'F{i}' for i in range(1,7)]
        assert all(x['selected_by'] is None for x in zs[6:])
        assert all(x['place'] in ZONE_CANDIDATES for x in zs)
        recount={k:sum(k in x['covered_functions'] for x in zs) for k in FUNCTION_QUOTAS}
        assert recount==z['function_counts']
        if not z['missing_minimum']:counts['met_all_minimum']+=1
        if not z['excess_maximum']:counts['met_all_maximum']+=1
        for k,v in z['missing_minimum'].items():miss[k]+=1
        for k,v in z['excess_maximum'].items():over[k]+=1
        places.update(x['place'] for x in zs)
        compositions.add(tuple(sorted(x['place'] for x in zs)))
        counts['worlds']+=1
        counts[f"count_{len(zs)}"]+=1
        counts['total_excess_instances']+=sum(z['excess_maximum'].values())
        if len(examples)<5:examples.append(w)
    return {'worlds':n,'met_all_minimum':counts['met_all_minimum'],
            'met_all_maximum':counts['met_all_maximum'],
            'zone_count_distribution':{k:counts[k] for k in ('count_7','count_8','count_9')},
            'different_zone_sets':len(compositions),'unique_zone_types':len(places),
            'top_places':places.most_common(15),'bottom_places':places.most_common()[-12:],
            'missing_minimum_worlds_by_function':dict(miss),
            'over_maximum_worlds_by_function':dict(over),
            'total_excess_instances':counts['total_excess_instances'],
            'examples':examples}


## Locked validation and scope

Source: Infaction_Zone_5_40_Tuning_10k_Heldout.json; 10,000 worlds, seeds 14,900–24,899. Range 5.33%–38.85%; 0 types outside 5%–40%; effective diversity 31.87/36; min Function compliance 99.95%, max compliance 100%. This is a probabilistic statistical target, NOT a hard limit per seed/batch.

Run zone_qa(n) and the export cells below for fresh live results. The reference figures above are archived results from a paired-seed vectorized implementation and should be cross-checked against the notebook implementation.


In [ ]:
# Run Zone simulation and present its audit.
ZONE_QA = zone_qa(10000)
print('ZONE QA:',{k:v for k,v in ZONE_QA.items() if k!='examples'})
for world in ZONE_QA['examples']:
    print('\nSEED',world['seed'],'ZONES',world['zone_generation']['target'],'MISSING',world['zone_generation']['missing_minimum'],'EXCESS',world['zone_generation']['excess_maximum'])
    for z in world['zone_generation']['zones']:
        print(f"  {z['sequence']} {z['selected_by'] or 'Extra'} — {z['place']} ({z['score']:.1f})")


In [ ]:
# Export full Faction + Zone examples and audit for downstream Colab use.
ZONE_EXAMPLES = [generate_world_with_zones(s) for s in [20261120,20261121,20261122,20261123,20261124]]
pathlib.Path('infaction_v0.4.1_faction_zones_5_worlds.json').write_text(json.dumps(ZONE_EXAMPLES,indent=2,ensure_ascii=False))
pathlib.Path('infaction_v0.4.1_zone_qa_10000.json').write_text(json.dumps({k:v for k,v in ZONE_QA.items() if k!='examples'},indent=2,ensure_ascii=False))
print('Saved Zone examples and QA.')


In [ ]:
# Quick multiple-world demonstration (six final Factions each).
DEMO_SEEDS=[20261120,20261121,20261122,20261123,20261124]
for demo_seed in DEMO_SEEDS:
    result=generate_world(demo_seed)
    print(f'\nWORLD SEED {demo_seed}')
    print('ID | Types | Base | Animal | Grammar | Final Faction Name')
    for f in result['factions']:
        print(f"{f['id']} | {' + '.join(f['types'])} | {f['base']} | {f['animal']} | {f['grammar']} | {f['name']}")


In [ ]:
# Structural QA and diversity metrics (rerunnable in Colab).
def qa(n=10000):
    stats=Counter(); distinct_types=Counter(); names=Counter(); bases=Counter(); animals=Counter()
    for seed in range(n):
        world=generate_world(seed)
        fs=world['factions']
        assert len(fs)==6 and [f['id'] for f in fs]==[f'F{i}' for i in range(1,7)]
        assert fs[0]['types'][0]=='Government' and fs[1]['types'][0]=='Revolutionary'
        assert all(1<=len(f['types'])<=2 for f in fs)
        assert all('Government' not in f['types'] or not (set(f['types']) & {'Elite','Poor','Revolutionary','Outsider'}) for f in fs)
        assert len({t for f in fs for t in f['types']}) == sum(map(lambda f:len(f['types']),fs))
        assert len({f['name'] for f in fs})==6
        assert len({f['base'] for f in fs})==6
        assert len({f['animal'] for f in fs})==6
        assert all(BASE[f['base']]['grammar'][f['grammar']]>0 for f in fs)
        # Base and Animal remain selected strictly by highest available score.
        # Full rankings can be inspected on an individual seed with retain_rankings=True.
        stats['worlds']+=1
        stats['active_11_types']+=(sum(len(f['types']) for f in fs)==11)
        stats['active_12_types']+=(sum(len(f['types']) for f in fs)==12)
        active_types=tuple(sorted(t for f in fs for t in f['types']))
        distinct_types[active_types]+=1
        for f in fs:
            names[f['name']]+=1; bases[f['base']]+=1; animals[f['animal']]+=1
    return {'worlds':n,'active_type_11':stats['active_11_types'],
            'active_type_12':stats['active_12_types'],
            'distinct_type_sets':len(distinct_types),
            'distinct_names':len(names),'distinct_bases':len(bases),
            'distinct_animals':len(animals),
            'top_bases':bases.most_common(10),'top_animals':animals.most_common(10)}
qa_result=qa(10000)
print('QA PASS',qa_result)
assert generate_world(20261120)==generate_world(20261120)


In [ ]:
# Export example world and batch samples. Only six final Factions appear anywhere in the data.
SEED=20261120
world=generate_world(SEED)
json_path=pathlib.Path(f'infaction_factions_v{VERSION}_seed_{SEED}.json')
json_path.write_text(json.dumps(world,ensure_ascii=False,indent=2))
tsv_path=pathlib.Path(f'infaction_factions_v{VERSION}_seed_{SEED}.tsv')
with tsv_path.open('w',newline='') as stream:
    writer=csv.writer(stream,delimiter='\t')
    writer.writerow(['slot','type_1','type_2','base','base_score','animal','animal_score','grammar','name'])
    for f in world['factions']:
        writer.writerow([f['id'],f['types'][0],f['types'][1] if len(f['types'])==2 else '',f['base'],f['base_score'],f['animal'],f['animal_score'],f['grammar'],f['name']])
print('Saved:',json_path,tsv_path)
